## Cell 0. API keys

Paste your keys between the quotes below and run this cell before anything
else. Leave a line as `""` to use whatever is already exported in the
environment instead.

**Two things worth knowing before you paste.** This notebook is regenerated by
`build_q1_nb.py`, which rewrites every cell from source -- so a key typed here
is lost on the next rebuild. And a key typed here is saved inside the `.ipynb`
file, where it can reach git or a shared copy. For a key you intend to keep,
put it in `fourarm/env/keys.local.env` instead, which this cell reads
automatically and which is gitignored and never regenerated.

In [1]:
# --- Cell 0. API keys. Run first. -------------------------------------------
import os, pathlib

# PASTE BETWEEN THE QUOTES. Leave "" to fall back to the environment or to
# env/keys.local.env.
KEYS = {
    "OPENAI_API_KEY": "",
    "GEMINI_API_KEY": "",
    "ANTHROPIC_API_KEY": "",
}

# An EMPTY value must never be written into the environment. Assigning ""
# unconditionally would blank a key that is already exported correctly, and
# the failure -- a 401 from a variable that is set but empty -- reads nothing
# like "you left the placeholder alone".
for _name, _value in KEYS.items():
    if _value.strip():
        os.environ[_name] = _value.strip()

# The persistent alternative. Same KEY=value format as env/models.env, one
# per line, # for comments. Read only for names not already set, so anything
# pasted above and anything already exported both win over the file.
_here = pathlib.Path.cwd()
_root = next((c for c in [_here] + list(_here.parents)
              if (c / "out").is_dir() and (c / "experiments").is_dir()), None)
_local = _root / "env" / "keys.local.env" if _root else None
if _local and _local.exists():
    for _line in _local.read_text().splitlines():
        _line = _line.strip()
        if not _line or _line.startswith("#") or "=" not in _line:
            continue
        _k, _, _v = _line.partition("=")
        _k, _v = _k.strip(), _v.strip().strip("\'\"")
        if _v and not os.environ.get(_k):
            os.environ[_k] = _v

# Report presence, NEVER the value. Printing a key would write it into the
# notebook's saved output, which is the same leak as pasting it into a cell
# and is easier to do by accident.
#
# The report loops over KEYS, so EVERY key the notebook can use needs a row
# there even when it is only ever supplied by keys.local.env. A name missing
# from KEYS still loads from the file, but silently, and a key that loads
# without being reported is indistinguishable from one that did not load.
for _name in KEYS:
    _set = bool(os.environ.get(_name))
    print("%-18s %s" % (_name, "set" if _set else "NOT SET"))
if _local:
    print("%-18s %s" % ("keys.local.env",
                        "read" if _local.exists() else "absent (optional)"))

OPENAI_API_KEY     set
GEMINI_API_KEY     set
ANTHROPIC_API_KEY  set
keys.local.env     read


# Experiment 2, Q3: Remediation

**Which kind of instruction moves a model from following the text to using the
scene?**

The six rungs are already defined in `prompts.RUNGS` and are not redefined
here. `N0` adds nothing, `N-A` directs attention, `N-C` states the derivation,
`N-order` changes the field order and nothing else, `N-D` forces the report
before the arm, and `N-CD` does C and D together.

**One repeat per rung**, against `N0` baselines collected at three. A rung's
per-position share is therefore 0 or 100, while the baseline's is one of four
values, so the second-order contrast is noisier than the first-order ones in
Q1 and Q2. That is a deliberate trade: five rungs across two conditions at
three repeats would be six thousand calls.

**The spend is staged and gated.** `N-D` runs first, in `dims`, three models,
one repeat. Everything after it is gated on what that shows, per model.

Cells 1 to 5 and 11 to 16 are free. Cells 6, 8, 9 and 10 spend.

In [2]:
# --- Cell 1. Setup. No model calls. -----------------------------------------
import collections, csv, datetime, hashlib, json, math, os, pathlib, sys

# Find the package root: the directory holding out/ and experiments/.
here = pathlib.Path.cwd()
ROOT = None
for cand in [here] + list(here.parents):
    if (cand / "out").is_dir() and (cand / "experiments").is_dir():
        ROOT = cand
        break
if ROOT is None:
    raise SystemExit("run this from fourarm/ or below: no out/ + experiments/ found")
for p in (str(ROOT), str(ROOT / "ycb")):
    if p not in sys.path:
        sys.path.insert(0, p)

# RE-IMPORT, never reuse. Python caches modules in sys.modules, so running
# this cell a second time in a live kernel keeps whatever was on disk the
# FIRST time it ran. While the ex2 modules are being edited alongside the
# notebook that is a trap: the kernel holds the old vocabulary, and the
# failure surfaces cells later as a design-check assertion naming a face
# that no longer exists, which reads like a code error and is not one.
#
# Dropping the entries and importing fresh is used rather than
# importlib.reload because these modules import each other, and reload
# leaves a half-updated graph unless the order is exactly right.
for _stale in [m for m in list(sys.modules)
               if m.startswith(("experiments.ex2", "analysis.ex2"))
               or m in ("ycb_objects",)]:
    del sys.modules[_stale]

# WHICH KERNEL THIS IS, checked before the first project import.
#
# The very next line reaches core.decision.state_builder through
# mancheck -> vlm_allocator, and that imports numpy; visibility.py, in cell
# 3, needs PIL and scipy. On a kernel without them the notebook dies forty
# lines deep inside somebody else's module with "No module named 'numpy'",
# which reads as a broken repository rather than as a kernel picked from a
# list of six. Checked here, where the answer is one sentence.
_missing = []
for _m in ("numpy", "PIL", "scipy"):
    try:
        __import__(_m)
    except ImportError:
        _missing.append(_m)
if _missing:
    _venv = ROOT.parent / ".venv" / "bin" / "python"
    raise SystemExit(
        "WRONG KERNEL.\n"
        "  This kernel is  %s\n"
        "  and it has no %s.\n"
        "  Use instead     %s\n"
        "  In VS Code: Select Kernel, then Python Environments, then the\n"
        "  interpreter at that path. It is the only one in this tree with\n"
        "  ipykernel AND numpy, PIL and scipy. Several unrelated kernels are\n"
        "  registered on this machine and any of them will get this far and\n"
        "  then fail."
        % (sys.executable, ", ".join(_missing), _venv))

from core.cell import cell_config as C
from core.decision import model_registry as MR
from experiments.ex2 import grade as G
from experiments.ex2 import labels as L
from experiments.ex2 import mancheck as MC
from experiments.ex2 import prompts as P
from experiments.ex2 import run as R
from experiments.ex2 import solo as S
from experiments.ex2 import transforms as T
from experiments.ex2 import visibility as VIS
from analysis.ex2.ex2_stats import newcombe, paired_mean_ci, spans_zero, wilson
# The notebook machinery: loaders, the share definition, the paired
# contrast and the spend gate. In a module rather than in this cell so
# that Q2 and Q3 use the same ones rather than a second copy, and so
# that harness/h_ex2_q_common.py can pin them. What stays in the cells
# is what is a DECISION: the models, the rung, the conditions, the
# usable rule, each cost, and every CONFIRM_SPEND.
from analysis.ex2.ex2_q_common import (Outputs, answered,       # noqa
                                       coupling, fmt, full_flip_count,
                                       is_franka, keep_analysable,
                                       load_run, paired_delta,
                                       paired_diffs, pct,
                                       provenance_row, run_meta,
                                       sha256, share_at, share_counts,
                                       show, spend_gate)

# --- paths ------------------------------------------------------------------
CAPTURES = ROOT / "out" / "ex2_capture_block"
RUNS     = ROOT / "runs"
TABLES   = ROOT / "tables" / "ex2_q3"
FIGURES  = ROOT / "figures" / "ex2_q3"
for d in (RUNS, TABLES, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

# --- constants, every one read from a source of truth ------------------------
RUNG        = "N0"                       # the BASELINE rung. The ladder
                                         # is cell 4, from prompts.RUNGS
PREFERENCE  = "franka"
CONDITIONS  = ("dims", "conflict_face")
REPEATS     = 3
FACES       = P.RESTING_FACES            # small_face, large_face
LABEL       = "ycb_block"

FRANKA_MAX  = C.ARM_TYPES["franka"]["max_grasp_m"]
UR_MAX      = C.ARM_TYPES["ur10"]["max_grasp_m"]
DIMS        = T.DIMS_M[LABEL]
FACTS       = T.POSE_FACTS_BY_LABEL[LABEL]

# The registry has no hardcoded model list: aliases() reads FOURARM_MODELS.
try:
    ALIASES = MR.aliases()
except Exception as exc:
    ALIASES = []
    print("model registry unavailable (%s); set MODELS by hand below" % exc)
# THREE models since 2026-08-27. claude-sonnet-5 was added because the
# design needs a third model that CLEARS the two-way face probe: with two
# models, a single failure at cell 5b leaves one, and one model cannot show
# that a result is a property of models rather than of this one model.
# It is not here for being the most capable available; see env/models.env.
#
# claude_md RATHER THAN claude. Same model, claude-sonnet-5, at effort
# medium instead of the API default of high. At the default it read the
# two-way face probe at 65 percent against gpt's 95 and gemini's 100, and
# it failed by BIAS rather than blindness: large_face on 75 percent of
# trials, 90 percent right when the block lies flat and 40 percent when it
# stands. Deliberation is how a prior like "blocks lie flat" gains weight,
# so lower effort is the move that fits the failure. Cell 5b is what tests
# it. The default-effort runs stay on disk under the alias "claude".
#
# gpt_hi RATHER THAN gpt. Same model, gpt-5.6-terra, at reasoning_effort
# high instead of low. Claude runs at the Anthropic default effort of high
# and Gemini Flash exposes no effort control at all, so gpt at low made the
# one model with the LEAST test-time compute the yardstick for the other
# two. Effort is still not matched across providers and cannot be -- that
# stays in Limitations -- but the reasoning models are now on the same
# nominal tier.
#
# The low-effort runs are NOT deleted. runs/ex2_q1_cue2way_gpt_r*.jsonl
# record gpt at reasoning_effort low over this same sample and stay on disk
# as the evidence for what effort was worth here: 95 percent at low. A
# separate alias rather than an edit to GPT_PARAMS is what makes those rows
# still readable, which is the reason env/models.env gives for gpt_hi
# existing at all.
#
# Named rather than taken wholesale from ALIASES. FOURARM_MODELS also lists
# qwen and gpt_hi, and a run's model set must be a decision recorded here,
# not whatever the registry happens to carry. The fallback keeps the same
# three so a registry failure cannot silently shrink the design.
_WANT = ("gpt_hi", "gemini", "claude_md")
MODELS = tuple(a for a in ALIASES if a in _WANT) or _WANT
if set(MODELS) != set(_WANT):
    print("WARNING: %s requested, %s available from the registry. Every "
          "table below is per model, so a missing one narrows the design "
          "rather than breaking it -- but say so in the chapter."
          % (list(_WANT), list(MODELS)))

# --- credentials: reported, not assumed -------------------------------------
# Until 2026-08-27 a hand-added launcher cell started JupyterLab in a browser
# and refused to launch when a key was missing. That cell is gone: it spawned
# a NEW server every time it ran, which is how eight of them accumulated, each
# serving its own in-memory copy of this notebook, so an edit on disk could be
# invisible in the tab you were typing in. VS Code runs the kernel directly
# and needs no launcher -- but the key check it performed was worth keeping,
# so it lives here.
#
# This REPORTS rather than raises. Cells 1-5 and every analysis cell make no
# model calls and must stay runnable with no key at all. What it buys is
# learning about a missing key now instead of at cell 6, part-way into a run.
#
# The usual cause is launching VS Code from Finder or the Dock, which does not
# inherit a login shell, so a key exported in .zshrc is absent here while
# present in any terminal. The message below says so, because the symptom
# otherwise looks like a broken registry.
#
# key_var is read from the registry, never hardcoded: models.env lets each
# alias name its own variable, and a hardcoded OPENAI_API_KEY would check the
# wrong one the moment that is used.
MISSING_KEYS = []
for _alias in MODELS:
    try:
        _var = MR.describe(_alias)["key_var"]
    except Exception as _exc:
        MISSING_KEYS.append("%s: %s" % (_alias, _exc))
        continue
    if not os.environ.get(_var):
        MISSING_KEYS.append("%s: %s is not set" % (_alias, _var))

# Output paths travel together in one object, so a notebook cannot end up
# with a root and a tables directory that disagree. Rebound to bare names
# because every call site below reads better as write_csv(...) than as
# OUT.write_csv(...), and because leaving those call sites untouched is
# what made this extraction verifiable against the tables already on disk.
OUT = Outputs(ROOT, TABLES, FIGURES)
rel, write_csv = OUT.rel, OUT.write_csv

print("root        ", ROOT)
print("captures    ", CAPTURES.relative_to(ROOT), "(exists:", CAPTURES.is_dir(), ")")
print("rung        ", RUNG, " preference", PREFERENCE, " repeats", REPEATS)
print("models      ", MODELS, " (registry knows: %s)" % (ALIASES or "nothing"))
print("prompt ver  ", P.EX2_PROMPT_VERSION)
# Printed, not assumed. If a stale kernel ever slips past the re-import
# above, this is the line that shows it, at the top of the run rather than
# in an assertion twenty cells later.
print("faces       ", FACES, " chance %.1f%%" % (100.0 / len(FACES)))
if MISSING_KEYS:
    print("api keys     MISSING -- analysis runs, model calls will not:")
    for _m in MISSING_KEYS:
        print("               ", _m)
    print("             launch VS Code from a shell that exports them:")
    print("               open -a 'Visual Studio Code' <repo>")
else:
    print("api keys     present for %s" % (", ".join(MODELS),))
print()
print("block           %.3f x %.3f x %.3f m"
      % (DIMS["height"], DIMS["width"], DIMS["depth"]))
print("franka opens to  %.3f m   ur opens to %.3f m" % (FRANKA_MAX, UR_MAX))
print("resting faces   %s" % (FACES,))
for f in FACES:
    print("   %-11s needs %.3f m" % (f, FACTS[f]["grasp_m"]))

root         /Users/erinsarlak/Downloads/MastersDissertation/fourarm
captures     out/ex2_capture_block (exists: True )
rung         N0  preference franka  repeats 3
models       ('gpt_hi', 'gemini', 'claude_md')  (registry knows: ['qwen', 'gpt', 'gpt_hi', 'gemini', 'claude', 'claude_md'])
prompt ver   2026-08-27b
faces        ('small_face', 'large_face')  chance 50.0%
api keys     present for gpt_hi, gemini, claude_md

block           0.130 x 0.100 x 0.050 m
franka opens to  0.080 m   ur opens to 0.140 m
resting faces   ('small_face', 'large_face')
   small_face  needs 0.050 m
   large_face  needs 0.100 m


## Cell 2. Design check

No model calls. Derives the opening for each resting face from the authored
cuboid dimensions and asserts it matches what `transforms` declares. The prompt
states the bounding-box convention, so a divergence here would make the prompt
wrong rather than silent.

In [3]:
# --- Cell 2. Design check. No model calls. ----------------------------------
from ycb_objects import YCB as _SPECS      # the authored object dictionary

# STALE-IMPORT GUARD. Cell 1 purges sys.modules before importing, so a
# module edited on disk is picked up whenever cell 1 is re-run. This
# catches the case where cell 1 was NOT re-run -- editing a module and
# jumping straight back to this cell -- and the worse case where the
# NOTEBOOK ITSELF is stale, because JupyterLab holds its own copy in the
# browser and does not re-read the file when it changes underneath. A
# stale cell 1 has no purge, so the modules stay old and the design
# assertion below fails naming a face that no longer exists. That reads
# like a code error and is not one, which is why this checks first and
# says which of the two it is.
#
# The comparison is against the SOURCE ON DISK, not against a constant
# written here, so it stays true across future vocabulary changes.
import re                                   # local: a stale Cell 1 may
                                            # not have imported it
_src = pathlib.Path(P.__file__).read_text()
_on_disk = re.search(r'EX2_PROMPT_VERSION\s*=\s*["\'](.+?)["\']', _src)
if _on_disk and _on_disk.group(1) != P.EX2_PROMPT_VERSION:
    raise SystemExit(
        "STALE IMPORT: this kernel holds prompts.py version %s, but the file "
        "on disk is %s.\n"
        "  Loaded faces: %s\n"
        "  Fix: re-run Cell 1, which drops the cached modules and imports "
        "fresh.\n"
        "  If re-running Cell 1 does not clear it, the NOTEBOOK is stale, not "
        "the kernel:\n"
        "  JupyterLab is running the copy it loaded into the browser. Use "
        "File > Reload\n"
        "  Notebook from Disk, then Restart Kernel and Run All."
        % (P.EX2_PROMPT_VERSION, _on_disk.group(1), list(FACES)))

# Each block prim is spawned already resting on a face, with size PRE-ORIENTED
# to that pose: size is (x, y, z) with z vertical. So the two horizontal
# extents are size[0] and size[1], and the opening is the smaller of them.
# YCB is keyed without the "ycb_" scene prefix.
PRIM_OF_FACE = {L.TRUE_POSE[p]: p for p, lab in L.POSE_ENTRIES.items()
                if lab == LABEL}

design_rows = []
problems = []
for face in FACES:
    prim = PRIM_OF_FACE[face]
    size = _SPECS[prim.replace("ycb_", "")]["size"]
    horiz = sorted(size[:2], reverse=True)          # a = larger, b = smaller
    vertical = size[2]
    opening = min(horiz)
    declared = FACTS[face]["grasp_m"]

    if abs(opening - declared) > 1e-9:
        problems.append("%s: bounding box gives %.3f, transforms declares %.3f"
                        % (face, opening, declared))
    # A real permutation check, all three extents. It compared only the
    # SMALLEST until 2026-08-27, so a prim sized 0.200 x 0.200 x 0.050 --
    # not the block at all -- passed a check whose message said it was
    # verifying a permutation. That matters more with two faces than it
    # did with three: there are fewer cross-checks left, and this cell is
    # what stands between a mis-authored prim and the whole experiment.
    if (sorted(round(v, 6) for v in list(horiz) + [vertical])
            != sorted(round(DIMS[k], 6) for k in ("height", "width", "depth"))):
        problems.append(
            "%s: extents %s are not a permutation of the block %s"
            % (face, sorted(list(horiz) + [vertical]),
               sorted(DIMS[k] for k in ("height", "width", "depth"))))

    franka_ok = declared <= FRANKA_MAX
    ur_ok = declared <= UR_MAX
    design_rows.append([face, "%.3f" % vertical, "%.3f" % horiz[0],
                        "%.3f" % horiz[1], "%.3f" % declared,
                        "%.3f" % FRANKA_MAX, "%.3f" % UR_MAX,
                        franka_ok, ur_ok,
                        "franka, preference satisfied" if franka_ok
                        else "UR, preference overridden"])

# The design only works if the Franka is feasible on one face and not the
# other, and the UR on both. Anything else and Q1 has no contrast.
feasible = [r[0] for r in design_rows if r[7]]
if sorted(feasible) != ["small_face"]:
    problems.append("franka feasible on %s, expected small_face alone"
                    % sorted(feasible))
if not all(r[8] for r in design_rows):
    problems.append("a UR is infeasible somewhere; it must be legal everywhere")

show(["face", "vert", "horiz_a", "horiz_b", "opening", "franka", "ur", "picks"],
     [[r[0], r[1], r[2], r[3], r[4], r[7], r[8], r[9]] for r in design_rows])
print()
if problems:
    raise AssertionError("DESIGN CHECK FAILED:\n  " + "\n  ".join(problems))
print("PASS  every opening is the smaller horizontal extent, and the Franka")
print("      is feasible on small_face and not on large_face.")
print("      A third face, the middle one, was withdrawn on 2026-08-27: it")
print("      was flat like large_face and differed only in geometry, which")
print("      made it the sharper test, but no model read it (GPT 58%,")
print("      Fisher p=0.76 over 81 trials). The cost is that a model")
print("      reading posture and applying a rule can no longer be told")
print("      apart from one deriving the opening from geometry.")

write_csv("tab_ex2_q3_geometry.csv",
          ["resting_face", "vertical_m", "horiz_a_m", "horiz_b_m",
           "opening_needed_m", "franka_max_m", "ur_max_m", "franka_feasible",
           "ur_feasible", "deriving_model_picks"],
          design_rows)

face        vert   horiz_a  horiz_b  opening  franka  ur    picks                       
----------  -----  -------  -------  -------  ------  ----  ----------------------------
small_face  0.130  0.100    0.050    0.050    True    True  franka, preference satisfied
large_face  0.050  0.130    0.100    0.100    False   True  UR, preference overridden   

PASS  every opening is the smaller horizontal extent, and the Franka
      is feasible on small_face and not on large_face.
      A third face, the middle one, was withdrawn on 2026-08-27: it
      was flat like large_face and differed only in geometry, which
      made it the sharper test, but no model read it (GPT 58%,
      Fisher p=0.76 over 81 trials). The cost is that a model
      reading posture and applying a rule can no longer be told
      apart from one deriving the opening from geometry.
wrote tables/ex2_q3/tab_ex2_q3_geometry.csv  (2 rows)


PosixPath('/Users/erinsarlak/Downloads/MastersDissertation/fourarm/tables/ex2_q3/tab_ex2_q3_geometry.csv')

## Cell 3. Capture inventory and legality

No model calls. Loads the captures, checks the grid is complete, re-asserts the
recorded settle heights, and runs the **real validator** over every scene to
establish which positions can carry the contrast at all.

This cell is the reason the sample is 29 positions rather than 30, and it fails
loudly rather than letting the analysis assume otherwise.

**Why the settle heights are re-checked here.** The prompt never says which flat
orientation to expect. The convention sentence -- *an object resting flat lies on
its largest face* -- was deliberately left out, because capture enforces it
instead: `capture_ex2_scene.py` fails a capture that settles at the wrong height
rather than relabelling it with the face it was asked for. That assertion is real
and it does raise, but it post-dates most of the captures on disk, and the trail
check below it compares only the recorded face *word* against the prim -- never
the height that word is supposed to describe. So the single guarantee standing
behind the prompt's silence was being taken on trust at the point where the data
is actually read. Every capture records `ex2.settled[name]`, so checking it costs
nothing. The tolerance is read out of the capture script's source rather than
typed here, so it cannot drift from the value the captures were accepted under.

In [4]:
# --- Cell 3. Capture inventory and legality. No model calls. ----------------
import re                                   # local, as in Cell 2: Cell 1
                                            # does not import it, so this
                                            # cell must not depend on Cell 2
                                            # having been run first
scenes = R.load_scenes(str(CAPTURES))          # normalises the idle UR
raw    = R.load_scenes(str(CAPTURES), present_ur=False)   # as written
trail  = [json.loads(l) for l in open(CAPTURES / "consults.jsonl") if l.strip()]

by_pos = collections.defaultdict(dict)
for s in scenes:
    pos, member = s["seq"].rsplit("_", 1)
    prim = [o["name"] for o in s["state"]["objects"] if LABEL.split("_")[-1] in o["name"]][0]
    by_pos[pos][L.TRUE_POSE[prim]] = s

# DERIVED, never literal. This read "90 captures / 30 positions" until
# 2026-08-27 and raised the moment four positions were added to the capture
# plan. A count typed here goes stale silently; one derived from the
# directory cannot. What actually matters is not the total but that every
# position carries every face, which `missing` below checks.
inv_problems = []
if len(scenes) != len(by_pos) * len(FACES):
    inv_problems.append("expected %d captures (%d positions x %d faces), "
                        "found %d" % (len(by_pos) * len(FACES), len(by_pos),
                                      len(FACES), len(scenes)))
missing = {p: sorted(set(FACES) - set(v)) for p, v in by_pos.items()
           if set(v) != set(FACES)}
if missing:
    inv_problems.append("positions missing a face: %s" % missing)
if len({s["seq"] for s in scenes}) != len(scenes):
    inv_problems.append("duplicate seq ids")

# The face is derived from the PRIM, never from the trail's word, and the
# trail is then checked against it.
#
# Captures written before 2026-08-27 record ex2.resting_face in a superseded
# vocabulary where "upright" meant small_face and "small_face" meant the
# retired middle face. capture_ex2_scene.py now writes the geometric name
# directly, so new captures need no translation; this map reads the old ones
# and is why the check is against the prim rather than the word.
TRAIL_FACE = {"upright": "small_face", "small_face": "edge",
              "large_face": "large_face"}

# READ FROM THE CAPTURE SCRIPT, not typed here. capture_ex2_scene.py
# imports isaaclab at module scope and cannot be imported into this kernel,
# and a literal copied into the notebook would go stale the moment the
# tolerance is retuned -- which it was, from 0.010 to 0.005, on 2026-08-27.
# Same regex-the-source trick cell 2 uses for EX2_PROMPT_VERSION.
_cap_src = (ROOT / "ycb" / "capture_ex2_scene.py").read_text()
_tol = re.search(r"^SETTLE_TOL_M\s*=\s*([0-9.]+)", _cap_src, re.M)
if not _tol:
    raise SystemExit(
        "SETTLE_TOL_M not found in ycb/capture_ex2_scene.py. It is the "
        "tolerance the captures were accepted under and the notebook must "
        "not invent one; if it was renamed, update this cell.")
SETTLE_TOL_M = float(_tol.group(1))
for rec in trail:
    prim = [o["name"] for o in rec["state"]["objects"] if "block" in o["name"]][0]
    want = L.TRUE_POSE.get(prim)
    if want is None:
        continue          # a retired-face capture; load_scenes drops it too
    word = (rec.get("ex2") or {}).get("resting_face")
    # BOTH vocabularies are accepted, and only because each is checked
    # against the prim. A word is fine if it already IS the derived face
    # (written 2026-08-27 or later) or if it translates to it (written
    # before). Anything else is a genuine disagreement. Accepting both is
    # not laxity: the prim is the truth in either case, and the word is
    # never the thing consulted downstream.
    if word != want and TRAIL_FACE.get(word) != want:
        inv_problems.append("%s: trail says %r, prim says %r"
                            % (rec["seq"], word, want))

    # THE SETTLE HEIGHTS, RE-ASSERTED WHERE THE DATA IS READ.
    #
    # The prompt says nothing about which flat orientation to expect. The
    # convention sentence ("an object resting flat lies on its largest
    # face") was deliberately NOT added, on the grounds that capture
    # enforces it instead -- and it does: capture_ex2_scene.py raises on a
    # capture that settles at the wrong height rather than labelling it
    # with the face it was asked for. But that assertion post-dates most
    # captures on disk, and the check above compares only the trail's face
    # WORD against the prim, never the height that word is supposed to
    # describe. So the one guarantee standing behind the prompt's silence
    # was, at this point, taken on trust. It is not expensive to check.
    for name, d in (rec.get("ex2") or {}).get("settled", {}).items():
        z, want_z = d.get("z_above_table"), d.get("expected_rest_z")
        if want_z is None:
            inv_problems.append("%s: %s has no expected_rest_z, so its "
                                "resting face was never verified"
                                % (rec["seq"], name))
        elif abs(z - want_z) > SETTLE_TOL_M:
            inv_problems.append(
                "%s: %s settled at z=%.4f, expected %.4f within %.3f. It is "
                "not on the face this capture claims."
                % (rec["seq"], name, z, want_z, SETTLE_TOL_M))

print("captures %d   positions %d   faces per position %s"
      % (len(scenes), len(by_pos),
         sorted({len(v) for v in by_pos.values()})))
print("idle UR presented: %s"
      % dict(collections.Counter(s["idle_ur"] for s in scenes)))
print("idle UR as captured: %s"
      % dict(collections.Counter(
          tuple(sorted(a["name"] for a in s["state"]["arms"]
                       if a["state"] == "IDLE" and a["name"].startswith("ur")))
          for s in raw)))
print()

# --- the real validator, per scene ------------------------------------------
legal = {}
for pos in sorted(by_pos):
    for face, s in by_pos[pos].items():
        st, meta = T.transform({"state": s["state"],
                                "positions_exact": s["positions_exact"]},
                               "congruent")
        tid = R.flip_task_id(s["state"], meta["flip_prim"])
        legal[(pos, face)] = sorted(R.legal_arms(s, meta["flip_prim"], tid))

def has_franka(arms):
    return any(a.startswith("franka") for a in arms)

# TWO independent preconditions, not one. Legality asks whether the aperture
# contrast EXISTS at a position; visibility asks whether the block can be
# SEEN there. A position can carry the full contrast with the block hidden
# behind the Franka, and until 2026-08-27 nothing noticed: e10 presents 3%
# of the median block area and GPT inverted both its posture trials.
#
# visibility.verdict reads pixels only and never a model reply, so a
# position is never excluded for having scored badly.
#
# Run PER PREFIX, not over the directory at once. Each pose is scored
# against the median of its own pose, and the west and east banks sit at
# different distances from the camera: a w block renders about 10 percent
# larger than an e block in the same pose. One pooled median would raise
# the bar for the far bank and lower it for the near one, which is a
# comparison between banks rather than a test of occlusion.
OCCLUDED, _vis_detail = [], {}
for _pfx in sorted({p[0] for p in by_pos}):
    _c = VIS.measure(str(CAPTURES), prefix=_pfx)
    _ok, _bad, _d = VIS.verdict(_c)
    print("visibility, %s bank:" % _pfx)
    print(VIS.report(_d, _ok, _bad))
    print()
    OCCLUDED += _bad
    _vis_detail.update(_d)

USABLE, excluded = [], {}
for pos in sorted(by_pos):
    sm, lg = (legal[(pos, f)] for f in ("small_face", "large_face"))
    ok = has_franka(sm) and lg and not has_franka(lg)
    if ok:
        USABLE.append(pos)
    else:
        excluded[pos] = {"small_face": sm, "large_face": lg}

show(["position", "small_face", "large_face", "usable"],
     [[pos, ",".join(legal[(pos, "small_face")]) or "NONE",
       ",".join(legal[(pos, "large_face")]) or "NONE",
       "yes" if pos in USABLE else "NO"] for pos in sorted(by_pos)])
print()
USABLE = [p for p in USABLE if p not in OCCLUDED]
print("positions carrying the full contrast and showing the block: %d of %d"
      % (len(USABLE), len(by_pos)))
for pos in sorted(set(OCCLUDED)):
    print("  EXCLUDED %s  the block is occluded here (%.2f of the pose"
          % (pos, _vis_detail[pos]["worst"]))
    print("           median, worst in %s). The contrast may exist, but a"
          % _vis_detail[pos]["worst_pose"])
    print("           perception result from a picture that does not show")
    print("           the object is not a result about the model.")
for pos, v in excluded.items():
    print("  EXCLUDED %s  %s" % (pos, v))
    print("           the franka is never legal here, so there is no arm choice")
    print("           to make and no contrast to measure. Excluded with cause,")
    print("           not dropped silently.")

write_csv("tab_ex2_q3_inventory.csv",
          ["position", "usable", "occluded", "idle_ur", "legal_small_face",
           "legal_large_face"],
          [[pos, pos in USABLE, pos in OCCLUDED,
            by_pos[pos]["small_face"]["idle_ur"],
            ";".join(legal[(pos, "small_face")]),
            ";".join(legal[(pos, "large_face")])] for pos in sorted(by_pos)])

if inv_problems:
    raise AssertionError("INVENTORY FAILED:\n  " + "\n  ".join(inv_problems))
if len(USABLE) < 20:
    raise AssertionError("only %d usable positions; the contrast is not "
                         "estimable and the run should not be paid for"
                         % len(USABLE))

# --- what the PAID cells ask about ------------------------------------------
# Defined here, ONCE, and used by both cell 6 and cell 7. Putting the choice
# in each paid cell would let the two be set differently, so congruent and
# dims would cover different scene sets and the paired contrast in cell 10
# would silently compare two different samples.
#
# TRUE is the standing decision: ask about every captured scene, including
# the excluded positions, and drop them in cell 8 at ANALYSIS time. It costs
# a little more and buys something the write-up needs -- the excluded rows
# are in the data, so the exclusion can be shown to predate any accuracy
# result rather than being read as a position dropped for scoring badly.
#
# Set FALSE to pay only for the usable positions. The analysis is unaffected
# either way: cell 8 restricts to USABLE regardless.
RUN_ALL_POSITIONS = True

CALL_SCENES = ([s for s in scenes
                if s["seq"].rsplit("_", 1)[0] in USABLE]
               if not RUN_ALL_POSITIONS else scenes)

print()
print("PASS  inventory complete, %d positions usable." % len(USABLE))
print("      paid cells will ask about %d scenes (%s)"
      % (len(CALL_SCENES),
         "all captured, excluded positions included on purpose"
         if RUN_ALL_POSITIONS else "usable positions only"))

[ex2] 34 capture(s) skipped: they rest on a face this design no longer uses, and are kept on disk as evidence. e00_S, e01_S, e02_S, e03_S ...
[ex2] 34 capture(s) skipped: they rest on a face this design no longer uses, and are kept on disk as evidence. e00_S, e01_S, e02_S, e03_S ...
captures 68   positions 34   faces per position [2]
idle UR presented: {'ur_w': 34, 'ur_e': 34}
idle UR as captured: {('ur_w',): 68}



visibility, e bank:
pos          L        S        U      worst  verdict
e10        224       51       45       0.03  OCCLUDED (U)
e05        488      893     1064       0.60  ok
e04        719     1380     1258       0.83  ok
e01        650     1273     1325       0.85  ok
e16        655     1311     1302       0.86  ok
e13        661     1327     1356       0.89  ok
e06        685     1438     1390       0.92  ok
e15        720     1483     1549       0.99  ok
e12        736     1497     1509       0.99  ok
e02        717     1499     1518       1.00  ok
e11        727     1495     1530       1.00  ok
e14        774     1551     1567       1.03  ok
e07        758     1562     1600       1.04  ok
e03        798     1564     1644       1.05  ok
e00        811     1650     1625       1.07  ok
e09        828     1709     1703       1.12  ok
e08        840     1741     1738       1.14  ok

cut at 0.50 of the pose median, in every pose.
worst excluded 0.03, best retained 0.60: the cut sits

visibility, w bank:
pos          L        S        U      worst  verdict
w01        646     1266     1326       0.84  ok
w10        663     1326     1374       0.88  ok
w16        669     1347     1363       0.88  ok
w07        679     1360     1369       0.89  ok
w05        687     1432     1424       0.92  ok
w15        719     1491     1503       0.97  ok
w02        720     1486     1517       0.97  ok
w13        720     1494     1541       0.97  ok
w11        768     1505     1507       0.98  ok
w14        741     1564     1563       1.00  ok
w04        758     1622     1595       1.02  ok
w03        787     1559     1587       1.03  ok
w00        802     1650     1620       1.05  ok
w06        800     1602     1621       1.05  ok
w09        803     1623     1627       1.06  ok
w12        824     1705     1717       1.11  ok
w08        855     1774     1757       1.14  ok

cut at 0.50 of the pose median, in every pose.
17 usable, 0 occluded: none
this reads pixels only and never a 

## Cell 4. The rung design

No model calls. The rungs come from `prompts.RUNGS`; this cell records what
each one adds, checks the factors are isolated, and prints the exact added
wording so the chapter quotes the prompt rather than a paraphrase of it.

In [5]:
# --- Cell 4. The rung design. No model calls. -------------------------------
# READ from the prompt module, never redefined. A rung list typed here would
# be a second source of truth for the thing the experiment manipulates.
LADDER = ("N0", "N-A", "N-C", "N-order", "N-D", "N-CD")
if set(LADDER) != set(P.PRE_REGISTERED_LADDER):
    raise SystemExit("the prompt module's pre-registered ladder is %s; this "
                     "notebook names %s"
                     % (sorted(P.PRE_REGISTERED_LADDER), sorted(LADDER)))

# THE CEILING, on the ladder but NOT pre-registered, and read from the module
# rather than typed: whatever LADDER_RUNGS holds beyond the six. It is kept
# out of LADDER on purpose. Cell 9d buys it and cell 9e reads it, so folding
# it in would widen cell 9's cost line and would put a rung added after the
# ladder was run into every table whose column says "factor". Naming it here
# is what stops it falling off the end of both lists and being swept by
# nothing, which the guard below then checks.
CEILING = tuple(r for r in P.LADDER_RUNGS if r not in LADDER)

# The off-ladder precedence directives, read from the module and never
# typed here. The second guard is what stops a variant being added to
# prompts.RUNGS and quietly reaching neither list: it would then be swept
# by nothing and checked by nothing, which is how a rung goes missing.
DIRECTIVES = tuple(P.DIRECTIVE_RUNGS)

# The Q3 repair treatments, read from the module for the same reason. They are
# a fourth category because prompts.py derives DIRECTIVE_RUNGS by exclusion
# from LADDER_RUNGS, so without REPAIR_RUNGS to name them they would land in
# it and be swept as precedence directives. This partition did not know they
# existed until 2026-09-10, and the guard below correctly refused to run.
REPAIR = tuple(P.REPAIR_RUNGS)

if (set(LADDER) | set(CEILING) | set(DIRECTIVES) | set(REPAIR)
        != set(P.RUNGS)):
    raise SystemExit("prompts.RUNGS holds %s, which is neither the ladder, "
                     "the ceiling, a declared directive, nor a repair rung"
                     % sorted(set(P.RUNGS) - set(LADDER) - set(CEILING)
                              - set(DIRECTIVES) - set(REPAIR)))

# WHICH DIRECTIVE CELL IS BOUGHT, IN STAGES, following the same discipline
# as GATE_RUNG below: a cheaper cell decides whether the rest is worth
# buying. Stage 2 exists only to INTERPRET stage 1 -- one control for "any
# extra sentence would have done it", one for "it merely heard the word
# image" -- so if stage 1 does not move there is nothing to control for and
# neither is bought.
DIRECTIVE_STAGE1 = (("conflict_face", "X-image"),)
DIRECTIVE_STAGE2 = (("congruent_face", "X-image"),
                    ("conflict_face", "X-state"))
DIRECTIVE_CELLS = DIRECTIVE_STAGE1 + DIRECTIVE_STAGE2

# THE GATE RUNG. N-D rather than N-CD, which is what the design document
# proposed. The pilot files from 2026-08-27 already show gemini going from
# 46.9 at N0 to a complete flip under N-D alone, with the face named
# correctly on every trial, and gpt not moving. So elicitation is the rung
# the evidence implicates, N-D minus N0 is the contrast that matters, and
# running the combined rung first would spend 204 calls confirming something
# a cheaper cell already indicates. N-CD becomes a sufficiency cell for a
# model N-D does not move.
GATE_RUNG = "N-D"
# Below this, at one repeat and 32 positions, an effect cannot be told from
# zero. Used to separate "not moved" from "not resolved", which are different
# findings and must not be reported as the same one.
GATE_MIN = 30.0

rung_rows = []
for rung in LADDER + CEILING:
    spec = P.RUNGS[rung]
    rung_rows.append([rung, ";".join(spec["factors"]) or "none",
                      spec["schema"],
                      len(spec["text"].strip().splitlines()),
                      spec["text"].strip().replace("\n", " ")[:60]
                      or "(nothing added)"])
show(["rung", "factors", "schema", "lines", "wording"], rung_rows)
write_csv("tab_ex2_q3_rungs.csv",
          ["rung", "factors", "schema", "added_lines", "added_wording"],
          [[r[0], r[1], r[2], r[3],
            P.RUNGS[r[0]]["text"].strip()] for r in rung_rows])

# The factors must be isolated, in every condition Q3 runs. These raise.
pp = []
for cond in CONDITIONS:
    try:
        P.assert_rungs_isolated(cond)
    except Exception as exc:
        pp.append("%s: %s" % (cond, exc))
try:
    P.assert_base_states_no_relation()
except Exception as exc:
    pp.append("base prompt: %s" % exc)
if pp:
    raise AssertionError("RUNG ISOLATION FAILED:\n  " + "\n  ".join(pp))
print()
print("PASS  every rung is N0 plus its own block at the same anchor, the")
print("      schema variants are as declared, and no rung leaks another's")
print("      wording. Checked in %s." % " and ".join(CONDITIONS))
print()
print("The table above is the LADDER and the CEILING. The directives are")
print("off it and are cell 4b: putting them in a table whose column says")
print("\"factor\" would present them as a fifth and sixth factor of a design")
print("that pre-registered four.")
if CEILING:
    print()
    print("NOT PRE-REGISTERED, and shown last: %s. It was added after the"
          % ", ".join(CEILING))
    print("ladder had been run, is bought by cell 9d and read by cell 9e, and")
    print("is deliberately not in LADDER: every cost line, every sweep and")
    print("every factor table here is the pre-registered six.")
print()
print("PRE-REGISTERED PREDICTIONS, from prompts.PREDICTIONS:")
for k, v in sorted(P.PREDICTIONS.items()):
    print("  %-16s %s" % (k, v))
print()
print("N-order exists because N-D moves the report ahead of the arm AND asks")
print("for the face, and a model generates left to right. Without the order")
print("control an N-D effect could not be attributed to either.")
print("N-CD is a sufficiency cell against N0. It is never an interaction")
print("test: this design is not powered for one and does not claim to be.")

rung     factors                                 schema        lines  wording                                                     
-------  --------------------------------------  ------------  -----  ------------------------------------------------------------
N0       none                                    base          0      (nothing added)                                             
N-A      attention                               base          2      The image shows the table as it is now. Look at the object i
N-C      derivation                              base          2      The opening an object needs is the smaller of its two horizo
N-order  order                                   report_first  0      (nothing added)                                             
N-D      elicitation;order                       face_first    2      Give "resting_face" and "opening_needed_m" BEFORE naming an 
N-CD     derivation;elicitation;order            face_first    5      The opening a

## Cell 4b. The precedence directives, off the ladder

No model calls. The ladder scaffolds the derivation while staying silent about
provenance. A **directive** does the opposite: it names the stated resting face
and says the image can contradict it, which the boundary rule forbids `A`, `C`
and `D`. So it is not a seventh rung. It measures whether a model has an
arbitration step at all that a direct instruction can reach, and it is the
ceiling on instructed arbitration in the way `congruent_face` is the ceiling on
derivation.

This cell prints the exact wording so the chapter quotes the prompt rather than
a paraphrase, and shows that the cell which cannot be interpreted cannot be
rendered either.

In [6]:
# --- Cell 4b. The precedence directives. No model calls. --------------------
dir_rows = []
for rung in DIRECTIVES:
    spec = P.RUNGS[rung]
    added = spec["text"][len(P.A_ATTEND):].strip()
    dir_rows.append([rung, ";".join(spec["factors"]), spec["schema"], added])
show(["directive", "factors", "schema", "sentence added to N-A"], dir_rows)
write_csv("tab_ex2_q3_directives.csv",
          ["directive", "factors", "schema", "full_wording",
           "sentence_added_to_N_A"],
          [[r[0], r[1], r[2], P.RUNGS[r[0]]["text"].strip(), r[3]]
           for r in dir_rows])

print()
print("EACH IS N-A PLUS ONE SENTENCE. That is the whole attribution: N-A is")
print("already on disk in conflict_face at %d repeats, so X minus N-A is the"
      % REPEATS)
print("precedence sentence and nothing else, bought for nothing.")
print()
print("BOTH SENTENCES NAME THE IMAGE and differ in one word. X-state is what")
print("separates reading the instruction from reacting to image-talk: if the")
print("model moves toward the picture under BOTH, it is not arbitrating.")
print()
print("THE WORDING IS CONDITIONAL, never assertive. \"The image disagrees with")
print("the text\" would be a FALSE sentence in congruent_face, and")
print("congruent_face is the control that makes the result readable: the same")
print("prompt, the same sentence, an antecedent that is never satisfied.")
print()
print("THE SCHEMA STAYS base. A directive that also reordered the answer would")
print("confound precedence with the report order, which is what N-order is")
print("there to separate. It also means the arm is committed BEFORE the")
print("opening is written, so a failure to move cannot be told apart from")
print("obedience arriving too late in the generation. That is a limitation of")
print("this cell and belongs in the chapter, not a defect to patch by giving")
print("the directive the face-first schema.")
print()
print("WHERE A DIRECTIVE IS REFUSED, from prompts.RUNG_VACUOUS_IN:")
for rung in DIRECTIVES:
    print("  %-8s vacuous in %s" % (rung, ", ".join(P.RUNG_VACUOUS_IN[rung])))
for cond in sorted(set(CONDITIONS) | {c for c, _ in DIRECTIVE_CELLS}):
    offered = [r for r in DIRECTIVES if r in P.rungs_for(cond)]
    print("  %-15s offers %s" % (cond, ", ".join(offered) or "(none)"))
try:
    P.system_prompt("X-image", "dims")
    raise AssertionError("dims rendered a directive; it states no face")
except ValueError as exc:
    print()
    print("PASS  rendering X-image in dims raises rather than buying it:")
    print("      %s" % " ".join(str(exc).split())[:66])
print()
print("dims pops resting_face from the state and its glossary says so, so a")
print("sentence about \"the stated resting face\" would name a field the prompt")
print("has just withdrawn. That is a comprehension puzzle, not the question")
print("under test, so the cell is unrenderable rather than merely discouraged.")

directive  factors                     schema  sentence added to N-A                                                 
---------  --------------------------  ------  ----------------------------------------------------------------------
X-image    attention;precedence_image  base    Where the image and the stated resting face disagree, go by the image.
X-state    attention;precedence_state  base    Where the image and the stated resting face disagree, go by the state.
wrote tables/ex2_q3/tab_ex2_q3_directives.csv  (2 rows)

EACH IS N-A PLUS ONE SENTENCE. That is the whole attribution: N-A is
already on disk in conflict_face at 3 repeats, so X minus N-A is the
precedence sentence and nothing else, bought for nothing.

BOTH SENTENCES NAME THE IMAGE and differ in one word. X-state is what
separates reading the instruction from reacting to image-talk: if the
model moves toward the picture under BOTH, it is not arbitrating.

THE WORDING IS CONDITIONAL, never assertive. "The image disagrees w

## Cell 5. What each rung actually adds

No model calls. The diff of every rung against `N0`, and the answer schema each
one asks for, so the wording and the field order are both on the record before
anything is bought.

In [7]:
# --- Cell 5. Rung prompts, diffed against N0. No model calls. ---------------
import difflib
for cond in CONDITIONS:
    print("=" * 70)
    print("CONDITION %s" % cond.upper())
    print("=" * 70)
    texts = P.rung_diff(cond)
    base = texts["N0"].splitlines()
    # rung_diff returns only the rungs coherent in this condition, so a
    # directive is simply absent where it is refused.
    for rung in LADDER + DIRECTIVES:
        if rung == "N0" or rung not in texts:
            continue
        added = [l[1:] for l in difflib.unified_diff(base,
                                                     texts[rung].splitlines(),
                                                     lineterm="", n=0)
                 if l.startswith("+") and not l.startswith("+++")]
        removed = [l[1:] for l in difflib.unified_diff(base,
                                                       texts[rung].splitlines(),
                                                       lineterm="", n=0)
                   if l.startswith("-") and not l.startswith("---")]
        print()
        print("%-8s schema %-12s +%d lines  -%d lines"
              % (rung, P.RUNGS[rung]["schema"], len(added), len(removed)))
        for l in added:
            if l.strip():
                print("   + " + l)
        for l in removed:
            if l.strip():
                print("   - " + l)
    print()

print("=" * 70)
print("THE ANSWER SCHEMA, per variant")
print("=" * 70)
for name in ("base", "report_first", "face_first"):
    used = [r for r in LADDER + DIRECTIVES if P.RUNGS[r]["schema"] == name]
    print()
    print("%-13s used by %s" % (name, ", ".join(used)))
    for line in P.schema_text(name).strip().splitlines():
        print("   " + line)
print()
print("resting_face is required ONLY by face_first. Asking for it at every")
print("rung would tell the model that the face matters, which is the thing")
print("factor D is there to manipulate.")

CONDITION DIMS

N-A      schema base         +3 lines  -0 lines
   + The image shows the table as it is now. Look at the object in the image
   + before you choose.

N-C      schema base         +3 lines  -0 lines
   + The opening an object needs is the smaller of its two horizontal extents,
   + so it depends on which face the object is resting on.

N-order  schema report_first +3 lines  -3 lines
   + Answer ONLY with JSON, no prose, with the fields in this order:
   +   "opening_needed_m": <number>,
   +   "basket": "<any box the arm you named can reach>"
   - Answer ONLY with JSON, no prose:
   -   "basket": "<any box the arm you named can reach>",
   -   "opening_needed_m": <number>

N-D      schema face_first   +7 lines  -3 lines
   + Give "resting_face" and "opening_needed_m" BEFORE naming an arm, and choose
   + the arm to fit the opening you gave.
   + Answer ONLY with JSON, no prose, with the fields in this order:
   +   "resting_face": "<small_face | large_face>",
   +   "ope

## Cell 6. The gate: N-D in dims

**Makes model calls.** 68 scenes, three models, one repeat. This is the rung
the 2026-08-27 pilots implicate, and everything after it is gated on what it
shows, per model.

In [8]:
# --- Cell 6. GATE: N-D in dims. MAKES MODEL CALLS. --------------------------
def rung_file(cond, rung):
    """One file per condition and rung. The rung is in solo's trial_id too,
    so this is belt and braces -- but a rung pointed at the wrong file would
    find every id present, skip the lot, and report itself complete having
    spent nothing."""
    return RUNS / ("ex2_q3_%s_%s.jsonl" % (cond, rung))

GATE_OUT = rung_file("dims", GATE_RUNG)

# SEEDED FROM THE PILOTS ALREADY ON DISK, before the cost is computed.
# Two files from 2026-08-27 hold exactly this cell for gpt_hi and gemini:
# same rung, same condition, same 68 scenes, same prompt version, one
# repeat, default face order and frame. They were written by solo.run, so
# their trial_ids are the ones this cell would generate, and copying them
# in means the runner resumes over them rather than buying them twice.
# That is 136 of the 204 calls.
#
# THE FILTER REBUILDS THE ID THIS CELL WOULD ASK FOR and takes only exact
# matches. That matters: the gemini file also holds a large_first arm,
# which is a different cell of a different factor, and its ids carry a
# suffix. Rebuilding rather than pattern-matching means a row from another
# arm cannot leak in, now or when another factor is added later.
PILOTS = (("runs/ex2_q1_dims_N-D_effort.jsonl", "gpt_hi"),
          ("runs/ex2_q1_dims_N-D_order.jsonl", "gemini"))

def seed_from_pilots(out_path, pilots, cond, rung):
    """Copy matching rows into out_path. Idempotent; returns what it added."""
    have = set()
    if pathlib.Path(out_path).exists():
        for line in open(out_path):
            if line.strip():
                have.add(json.loads(line).get("trial_id"))
    added, skipped = collections.Counter(), collections.Counter()
    with open(out_path, "a") as fh:
        for src, model in pilots:
            if not pathlib.Path(src).exists():
                skipped["source file missing"] += 1
                continue
            seen = {}
            for line in open(src):
                if line.strip():
                    r = json.loads(line)
                    seen[r.get("trial_id")] = r
            for r in seen.values():
                if r.get("model") != model or r.get("error"):
                    continue
                rep = r.get("repeat") or 1
                if rep > REPEATS:
                    skipped["beyond this cell's repeats"] += 1
                    continue
                want = "%s|%s|%s|%s|%s|V|r%d" % (r.get("seq"), cond, model,
                                                 PREFERENCE, rung, rep)
                if r.get("trial_id") != want:
                    skipped["a different factor arm"] += 1
                    continue
                if r.get("ex2_prompt_version") != P.EX2_PROMPT_VERSION:
                    skipped["older prompt version"] += 1
                    continue
                if want in have:
                    continue
                fh.write(json.dumps(r) + "\n")
                have.add(want)
                added[model] += 1
    return added, skipped

_added, _skipped = seed_from_pilots(GATE_OUT, PILOTS, "dims", GATE_RUNG)
if _added:
    print("seeded from the pilots: %s"
          % ", ".join("%s %d" % (m, n) for m, n in sorted(_added.items())))
    for _src, _m in PILOTS:
        print("   %s" % _src)
    print("   Not new observations: the same cell, already paid for, resumed")
    print("   rather than re-bought. Cell 16 records the provenance.")
if _skipped:
    print("   not seeded: %s"
          % ", ".join("%s %d" % (k, v) for k, v in sorted(_skipped.items())))

n_calls = len(CALL_SCENES) * len(MODELS) * REPEATS
print("COST: %d scenes x %d models x %d repeat = %d calls"
      % (len(CALL_SCENES), len(MODELS), REPEATS, n_calls))
print("      rung %s, condition dims, against the N0 baseline already on disk"
      % GATE_RUNG)

CONFIRM_SPEND = None            # <-- set to the number in the COST line

if spend_gate(n_calls, CONFIRM_SPEND, GATE_OUT,
              factors=(("scenes", len(CALL_SCENES)), ("models", len(MODELS)),
                       ("repeats", REPEATS))):
    S.run(str(CAPTURES), out_path=str(GATE_OUT), models=MODELS,
          conditions=("dims",), preferences=(PREFERENCE,),
          rungs=(GATE_RUNG,), modalities=("V",), kind="pair", repeats=REPEATS)
    print("answered now:", answered(GATE_OUT))

   not seeded: source file missing 2
COST: 68 scenes x 3 models x 3 repeat = 612 calls
      rung N-D, condition dims, against the N0 baseline already on disk
already answered: 612 of 612 in ex2_q3_dims_N-D.jsonl
set CONFIRM_SPEND = 612 in this cell to proceed

not confirmed; no calls made.


## Cell 7. Gate read-out

No model calls. Three outcomes per model, not two: **moved**, **not moved**, and
**unresolved**. At one repeat over 32 positions an effect below about 30 points
cannot be told from zero, and reporting that as a null would be reading a
finding out of an interval that never had the resolution to produce one.

In [9]:
# --- Cell 7. Gate read-out. No model calls. ---------------------------------
# Every N0 baseline on disk, not only the two the ladder runs in. The four
# face and number conditions are what the rung effects have to be read
# against -- a rung that lifts dims to +50 means one thing beside a
# congruent_face ceiling of +100 and another beside one of +6 -- so they
# belong in the inventory and in the contrast table even though no rung is
# planned in them.
N0_FILE = {"dims": RUNS / "ex2_q1_dims_N0.jsonl",
           "conflict": RUNS / "ex2_q2_conflict_N0.jsonl",
           "congruent": RUNS / "ex2_q1_congruent_N0.jsonl",
           "congruent_face": RUNS / "ex2_q1_congruent_face_N0.jsonl",
           "conflict_face": RUNS / "ex2_q2_conflict_face_N0.jsonl"}

# Shown at N0 only: they carry no ladder, so listing five missing rungs
# apiece would be ten rows of noise.
BASELINES = tuple(c for c in N0_FILE if c not in CONDITIONS)

def rung_rows_for(cond, rung):
    """Analysable rows for one condition and rung, from the right file."""
    path = N0_FILE[cond] if rung == "N0" else rung_file(cond, rung)
    rows, _ = load_run(path, cond, MODELS)
    rows = [r for r in rows if r.get("rung") == rung]
    return keep_analysable(rows, USABLE)

def contrast_pairs(cond, rung, model):
    rows = [r for r in rung_rows_for(cond, rung) if r["model"] == model]
    return paired_diffs(rows, USABLE, "small_face", "large_face")

GATE_VERDICT = {}
gate_rows = []
for model in MODELS:
    a = contrast_pairs("dims", GATE_RUNG, model)
    b = contrast_pairs("dims", "N0", model)
    m_r, _, _, n_r = paired_mean_ci([d for _, d in a])
    m_0, _, _, n_0 = paired_mean_ci([d for _, d in b])
    delta = paired_delta(a, b)
    mean, lo, hi, npos = paired_mean_ci([d for _, d in delta])

    if not npos:
        v = "NOT RUN"
    elif lo > 0:
        v = "MOVED"
    elif hi < 0:
        v = "MOVED BACKWARDS"
    elif hi < GATE_MIN:
        v = "NOT MOVED"
    else:
        v = "UNRESOLVED"
    GATE_VERDICT[model] = v
    gate_rows.append([model, fmt(m_0), fmt(m_r), npos, fmt(mean), fmt(lo),
                      fmt(hi), v])

show(["model", "N0", GATE_RUNG, "npos", "delta", "lo", "hi", "verdict"],
     gate_rows)
print()
print("delta is %s minus N0, paired within position at both levels." % GATE_RUNG)
print("A model reads UNRESOLVED when the interval still admits an effect of")
print("%.0f points or more. That is not a null: it is a cell that needs more" % GATE_MIN)
print("repeats before it can be called either way.")
print()
for model in MODELS:
    v = GATE_VERDICT[model]
    print("  %-9s %s" % (model, v))
    if v == "MOVED":
        print("             -> elicitation works for this model. Cell 8 tests")
        print("                whether it needs the picture; cell 9 asks")
        print("                which factor did it.")
    elif v == "NOT MOVED":
        print("             -> the strongest single instruction did nothing.")
        print("                Cell 9 runs N-CD for this model as the")
        print("                sufficiency cell; the intermediate rungs are")
        print("                very unlikely to move what N-D did not.")
    elif v == "UNRESOLVED":
        print("             -> more repeats on this cell before any further")
        print("                rung is bought for this model.")
    else:
        print("             -> cell 6 has not been run.")
# THE GATE NO LONGER NARROWS THE DESIGN. It was written to skip models the
# strongest rung did not move, on the grounds that a weaker one would buy a
# row of zeros. Two things killed that. It returned UNRESOLVED rather than
# NOT MOVED for two models of three, and excluding a model on an
# inconclusive verdict is not the same as excluding it on a null; and at one
# repeat the whole ladder is cheap enough that a complete factorial costs
# less than the argument about which cells to skip. So the verdicts above
# are read as INFORMATION, and every cell below asks every model.
PROCEED = list(MODELS)
print()
print("Every rung below is run for every model. The verdicts above are a")
print("read-out, not a filter: two of three came back UNRESOLVED, and a")
print("model dropped on an inconclusive gate would be missing from the")
print("chapter with no result of its own to show for it.")

model      N0     N-D    npos  delta  lo     hi    verdict  
---------  -----  -----  ----  -----  -----  ----  ---------
gpt_hi     2.1    3.1    32    1.0    -13.2  15.3  NOT MOVED
gemini     46.9   100.0  32    53.1   40.7   65.5  MOVED    
claude_md  -10.4  0.0    32    10.4   -7.5   28.4  NOT MOVED

delta is N-D minus N0, paired within position at both levels.
A model reads UNRESOLVED when the interval still admits an effect of
30 points or more. That is not a null: it is a cell that needs more
repeats before it can be called either way.

  gpt_hi    NOT MOVED
             -> the strongest single instruction did nothing.
                Cell 9 runs N-CD for this model as the
                sufficiency cell; the intermediate rungs are
                very unlikely to move what N-D did not.
  gemini    MOVED
             -> elicitation works for this model. Cell 8 tests
                whether it needs the picture; cell 9 asks
                which factor did it.
  claude_md NOT MO

## Cell 8. The control: N-D with no image

**Makes model calls.** The rung that moves a model has to be shown to move it
*towards the scene*, not towards a better guess from the text.

Q1's ablation answers this at `N0`; it cannot answer it at `N-D`, because a
schema effect only shows under the schema. If a model scores here, where the
picture is absent, the face-first result is a text artefact and the rung is
withdrawn.

In [10]:
# --- Cell 8. CONTROL: N-D, dims, no image. MAKES MODEL CALLS. ---------------
CONTROL_OUT = RUNS / ("ex2_q3_dims_%s_noimage.jsonl" % GATE_RUNG)

# Only the models the gate moved. There is nothing to control for in a model
# that did not move.
CONTROL_MODELS = tuple(PROCEED)
n_calls = len(CALL_SCENES) * len(CONTROL_MODELS) * REPEATS
print("COST: %d scenes x %d models x %d repeat = %d calls"
      % (len(CALL_SCENES), len(CONTROL_MODELS), REPEATS, n_calls))
print("      models the gate moved: %s" % (", ".join(CONTROL_MODELS) or "none"))
print("      Under dims the two faces are indistinguishable in text, so this")
print("      must land at zero. If it does not, %s is a text artefact." % GATE_RUNG)

CONFIRM_SPEND = None            # <-- set to the number in the COST line

if not CONTROL_MODELS:
    print("\nno model to control; nothing to do.")
elif spend_gate(n_calls, CONFIRM_SPEND, CONTROL_OUT,
                factors=(("scenes", len(CALL_SCENES)),
                         ("models", len(CONTROL_MODELS)),
                         ("repeats", REPEATS))):
    S.run(str(CAPTURES), out_path=str(CONTROL_OUT), models=CONTROL_MODELS,
          conditions=("dims",), preferences=(PREFERENCE,),
          rungs=(GATE_RUNG,), modalities=("A",), kind="pair", repeats=REPEATS)
    print("answered now:", answered(CONTROL_OUT))

COST: 68 scenes x 3 models x 3 repeat = 612 calls
      models the gate moved: gpt_hi, gemini, claude_md
      Under dims the two faces are indistinguishable in text, so this
      must land at zero. If it does not, N-D is a text artefact.
already answered: 612 of 612 in ex2_q3_dims_N-D_noimage.jsonl
set CONFIRM_SPEND = 612 in this cell to proceed

not confirmed; no calls made.


## Cell 9. The rest of the dims ladder

**Makes model calls.** `N-A`, `N-C`, `N-order` and `N-CD` in `dims`, for every
model, one repeat. `N0` is Q1's and `N-D` is cell 6's.

No gating. The gate returned UNRESOLVED for two models of three, and dropping a
model on an inconclusive verdict would leave it missing from the chapter with
no result of its own. At one repeat the complete factorial is cheap enough that
the argument about which cells to skip costs more than the cells.

`solo.run` resumes, so the Gemini cells already collected are not re-bought.

In [11]:
# --- Cell 9. The rest of the dims ladder. MAKES MODEL CALLS. ----------------
# Every rung except N0 (Q1's) and N-D (cell 6's), for every model. One
# repeat. solo.run resumes, so the gemini cells already on disk are not
# re-bought and this asks only for what is missing.
DIMS_REST = tuple(r for r in LADDER if r not in ("N0", GATE_RUNG))

n_calls = len(CALL_SCENES) * len(MODELS) * REPEATS * len(DIMS_REST)
print("COST: %d scenes x %d models x %d repeat x %d rungs = %d calls"
      % (len(CALL_SCENES), len(MODELS), REPEATS, len(DIMS_REST), n_calls))
print("      rungs: %s" % ", ".join(DIMS_REST))
for rung in DIMS_REST:
    _f = rung_file("dims", rung)
    print("      %-8s %3d of %d answered" % (rung, answered(_f),
                                             len(CALL_SCENES) * len(MODELS)))

CONFIRM_SPEND = None            # <-- set to the number in the COST line

if spend_gate(n_calls, CONFIRM_SPEND,
              factors=(("scenes", len(CALL_SCENES)), ("models", len(MODELS)),
                       ("repeats", REPEATS), ("rungs", len(DIMS_REST)))):
    for rung in DIMS_REST:
        out = rung_file("dims", rung)
        print("\n--- dims %s ---" % rung)
        S.run(str(CAPTURES), out_path=str(out), models=MODELS,
              conditions=("dims",), preferences=(PREFERENCE,), rungs=(rung,),
              modalities=("V",), kind="pair", repeats=REPEATS)
    print("\ndims ladder complete")

COST: 68 scenes x 3 models x 3 repeat x 4 rungs = 2448 calls
      rungs: N-A, N-C, N-order, N-CD
      N-A      612 of 204 answered
      N-C      612 of 204 answered
      N-order  612 of 204 answered
      N-CD     612 of 204 answered
set CONFIRM_SPEND = 2448 in this cell to proceed

not confirmed; no calls made.


## Cell 9b. The frame control: `dims` under `extents`

**Makes model calls.** The `dims` glossary states the extents "measured
**standing on its smallest face**". The block is 0.130 x 0.100 x 0.050, so its
smallest face is 0.100 x 0.050, and *standing on its smallest face* is the
`small_face` condition word for word.

That leaves two failures indistinguishable under the `named` frame: a model that
cannot read the pose from the image, and a model that read the orientation the
glossary stated and never treated it as a convention. Both predict `small_face`
on every trial. The frame is the only thing that separates them, because the
first predicts the wording makes no difference and the second predicts the
answer follows it.

`extents`, already in `prompts.DIMS_FRAMES`, renders the same three numbers as
"three extents largest first" and names no orientation. It **removes** step 1
rather than scaffolding it, which is what a control should do.

**Three rungs, not one.** `N0` is the baseline: what the frame is worth with no
instruction at all. `N-CD` is where the named ladder first moves. `N-ACD` is the
ceiling. Together they give the frame effect at three points on the instruction
axis, which is what the confound calls into question. `N-A`, `N-C`, `N-order`
and `N-D` are the factor-attribution cells and are not needed to show the
frame effect exists; cell 9b2 buys them under the same frame once cell 9c
says there is one.

**Matched repeats.** The named ladder is on disk at three repeats. A frame arm at
one repeat would widen every interval by about 1.6x and leave the clean arm
noisier than the confounded one it replaces, so the repeat count is read from
disk and asserted rather than taken from `REPEATS`.

In [12]:
# --- Cell 9b. FRAME CONTROL: dims under extents. MAKES MODEL CALLS. ---------
FRAME_ALT = "extents"
FRAME_RUNGS = ("N0", "N-CD", "N-ACD")     # baseline, partial, complete
FRAME_OUT = {g: RUNS / ("ex2_q3_dims_%s_%s.jsonl" % (g, FRAME_ALT))
             for g in FRAME_RUNGS}
FRAME_BASE = {g: (N0_FILE["dims"] if g == "N0" else rung_file("dims", g))
              for g in FRAME_RUNGS}

# The alternative frame is prompts' own, never a string invented here. If it
# is renamed this must fail rather than quietly re-buy the DEFAULT frame and
# report it as a control: a file compared with itself reads as a clean null
# and is the most expensive way to be wrong in this notebook.
if FRAME_ALT not in P.DIMS_FRAMES:
    raise AssertionError(
        "%r is not in prompts.DIMS_FRAMES (%s)."
        % (FRAME_ALT, ", ".join(P.DIMS_FRAMES)))

# WHAT THE FRAME ACTUALLY CHANGES, printed rather than described. The
# manipulation is only interpretable if it touches the glossary line and
# nothing else, and if it makes the SAME edit at every rung -- a frame that
# changed more under one instruction than another would confound the two.
import difflib
print("=" * 70)
print("THE FRAME EDIT: %s minus named, at every rung this cell buys" % FRAME_ALT)
print("=" * 70)
_edit = None
for g in FRAME_RUNGS:
    _a = P.system_prompt(g, "dims", dims_frame="named").splitlines()
    _b = P.system_prompt(g, "dims", dims_frame=FRAME_ALT).splitlines()
    _d = list(difflib.unified_diff(_a, _b, lineterm="", n=0))
    add = [l[1:] for l in _d if l.startswith("+") and not l.startswith("+++")]
    rem = [l[1:] for l in _d if l.startswith("-") and not l.startswith("---")]
    print()
    print("%-7s  -%d lines  +%d lines" % (g, len(rem), len(add)))
    for l in rem:
        if l.strip():
            print("   - " + l)
    for l in add:
        if l.strip():
            print("   + " + l)
    if _edit is None:
        _edit = (tuple(add), tuple(rem))
    elif (tuple(add), tuple(rem)) != _edit:
        raise AssertionError(
            "the %s edit differs between %s and %s. The frame contrast has "
            "to be the same manipulation at every rung, or the frame and the "
            "instruction cannot be told apart."
            % (FRAME_ALT, FRAME_RUNGS[0], g))
print()
print("PASS  the frame makes the same edit at all %d rungs, so a difference"
      % len(FRAME_RUNGS))
print("      between them is the instruction and not the wording.")
print()
print("The phrase that goes away is \"standing on its smallest face\". The")
print("block's smallest face IS the small_face condition, so under the named")
print("frame the glossary names one of the two answers.")
print()
print("=" * 70)
print("THE FULL PROMPT SENT, dims N0 under %s" % FRAME_ALT)
print("=" * 70)
print(P.system_prompt("N0", "dims", dims_frame=FRAME_ALT))
print("=" * 70)

# REPEATS IS READ FROM DISK, not from the cell-1 constant. Q3 declares one
# repeat; the dims ladder on disk carries three. Hard-coding either would let
# the frame arm drift out of match with the arm it is read against, and an
# unmatched control is worth less than no control.
def repeats_on_disk(path, rung):
    if not pathlib.Path(path).exists():
        return 0
    rows, _ = load_run(path, "dims", MODELS)
    return len({r.get("repeat") for r in rows if r.get("rung") == rung})

_named = {g: repeats_on_disk(FRAME_BASE[g], g) for g in FRAME_RUNGS}
if 0 in _named.values():
    print("The named-frame rungs this is read against are not all on disk:")
    for g in FRAME_RUNGS:
        print("  %-6s %-46s %s" % (g, rel(FRAME_BASE[g]),
                                   "%d repeats" % _named[g] if _named[g]
                                   else "MISSING"))
    print("Cells 9 and 9d buy them. Nothing to control against, nothing bought.")
elif len(set(_named.values())) != 1:
    raise AssertionError(
        "the named rungs carry different repeat counts (%s). The frame arm "
        "can only match one of them, and a control matched to some cells and "
        "not others cannot be read as one contrast." % _named)
else:
    FRAME_REPEATS = next(iter(_named.values()))
    n_calls = (len(CALL_SCENES) * len(MODELS) * FRAME_REPEATS
               * len(FRAME_RUNGS))
    print("COST: %d scenes x %d models x %d repeats x %d rungs = %d calls"
          % (len(CALL_SCENES), len(MODELS), FRAME_REPEATS, len(FRAME_RUNGS),
             n_calls))
    print("      repeats read from disk, matched to the named ladder.")
    print()
    _per = len(CALL_SCENES) * len(MODELS) * FRAME_REPEATS
    for g in FRAME_RUNGS:
        print("      %-6s %-40s %4d of %d answered"
              % (g, rel(FRAME_OUT[g]), answered(FRAME_OUT[g]), _per))
    print()
    print("      solo appends the frame to the trial_id when it is not the")
    print("      default, so these ids cannot collide with the named runs'")
    print("      even though scene, model, rung and repeat are identical.")
    print("      A part-run cell resumes: only the missing repeats are bought,")
    print("      and an unparseable reply is retried rather than skipped.")
    print()
    print("      EVERY MODEL, including gemini. Gemini reads the pose at 100%%")
    print("      under the named frame, so a frame effect in another model")
    print("      means nothing unless gemini stays put in the same run.")

    CONFIRM_SPEND = None        # <-- set to the number in the COST line

    if spend_gate(n_calls, CONFIRM_SPEND,
                  factors=(("scenes", len(CALL_SCENES)),
                           ("models", len(MODELS)),
                           ("repeats", FRAME_REPEATS),
                           ("rungs", len(FRAME_RUNGS)))):
        for g in FRAME_RUNGS:
            print("\n--- dims %s, %s frame ---" % (g, FRAME_ALT))
            S.run(str(CAPTURES), out_path=str(FRAME_OUT[g]), models=MODELS,
                  conditions=("dims",), preferences=(PREFERENCE,),
                  rungs=(g,), modalities=("V",), kind="pair",
                  repeats=FRAME_REPEATS, dims_frames=(FRAME_ALT,))
        print("\nframe control complete")

THE FRAME EDIT: extents minus named, at every rung this cell buys

N0       -4 lines  +3 lines
   - Each object states its mass, "mass_kg", whether it is "delicate", its height,
   - width and depth measured standing on its smallest face, "size_upright_m", and
   - the arms that can reach it, "arms_that_can_reach". No opening and no resting
   - face are given.
   + Each object states its mass, "mass_kg", whether it is "delicate", its three
   + extents largest first, "extents_m", and the arms that can reach it,
   + "arms_that_can_reach". No opening and no resting face are given.

N-CD     -4 lines  +3 lines
   - Each object states its mass, "mass_kg", whether it is "delicate", its height,
   - width and depth measured standing on its smallest face, "size_upright_m", and
   - the arms that can reach it, "arms_that_can_reach". No opening and no resting
   - face are given.
   + Each object states its mass, "mass_kg", whether it is "delicate", its three
   + extents largest first, "exte

## Cell 9b2. The rest of the ladder under `extents`

**Makes model calls.** Cell 9b buys the frame at three points on the instruction
axis: no instruction, the pair, the complete procedure. This buys the four rungs
it left out -- `N-A`, `N-C`, `N-order` and `N-D` -- so the frame contrast exists
at **every** rung of the ladder.

**Why the four are worth buying.** They are the single-factor cells, and they are
what cells 12 and 13 read to say *which* instruction moved the model. Under the
`named` frame every one of those attributions carries the same confound the frame
control exists to test: a factor that "moves the model" may be moving it off a
glossary that already named the answer. Bought here, each attribution can be read
again under a glossary that names no orientation, and a factor that survives both
frames is a factor about grounding rather than about wording.

**Read cell 9c before buying this.** It is the expensive cell -- four rungs at the
matched three repeats, a third more calls than 9b -- and 9b plus 9c is what says
whether it is worth anything. If the anchor does not move at `N0`, `N-CD` and
`N-ACD`, the named ladder is not being read off the wording and this cell re-buys
a null four more times. If it does move, the ladder's own attribution is in
question and these four are how it gets settled.

**Nothing about the control is restated here.** The frame name, the file naming
and the repeat rule are cell 9b's, used as it left them, so the eight files are
one arm rather than two arms that happen to share a suffix. The cell asserts the
`extents` edit is the same at all seven rungs before it spends, and reads the
repeat count off the named ladder exactly as 9b does.

**It widens `FRAME_RUNGS`**, so cell 9c's table and `tab_ex2_q3_frame.csv` cover
the whole ladder instead of the three. Re-running 9b narrows it back to its own
three; re-run this cell afterwards if that happens.

In [13]:
# --- Cell 9b2. FRAME CONTROL, THE REST OF THE LADDER. MAKES MODEL CALLS. ----
# The four rungs cell 9b left on the named frame: the single-factor cells.
# They are what cells 12 and 13 read to attribute the movement to a factor,
# so leaving them on the named frame leaves every attribution resting on a
# glossary that names one of the two answers. This buys them under the same
# frame 9b bought, at the same repeats, so the eight files are one arm.
#
# READ CELL 9c FIRST. If the anchor does not move at N0, N-CD and N-ACD then
# the ladder is not being read off the wording, and this cell buys the same
# null four more times.

# CELL 9b DEFINES THE ARM; this only widens it. The frame name, the output
# naming and the repeat rule are stated once, in 9b, so that this cell
# cannot buy a differently-defined control and file it beside the first --
# two arms under one name is worse than one arm and a gap.
for _n in ("FRAME_ALT", "FRAME_RUNGS", "FRAME_OUT", "FRAME_BASE",
           "repeats_on_disk"):
    if _n not in globals():
        raise AssertionError(
            "%s is not defined, so cell 9b has not been run in this kernel. "
            "This cell extends 9b's arm rather than restating it." % _n)

# WHICH RUNGS ARE 9b's, remembered the first time this cell runs. FRAME_RUNGS
# is widened below so that cell 9c reads the whole ladder, which means it can
# no longer be the thing that answers "which rungs did 9b buy". Re-running
# this cell is meant to be harmless -- solo.run resumes -- and it stays
# harmless because FRAME_CORE is not recomputed once it exists.
FRAME_CORE = globals().get("FRAME_CORE") or tuple(FRAME_RUNGS)
FRAME_REST = tuple(g for g in P.LADDER_RUNGS if g not in FRAME_CORE)
if not FRAME_REST:
    raise AssertionError(
        "cell 9b already names every ladder rung (%s), so there is no rest "
        "of the ladder for this cell to buy." % ", ".join(FRAME_CORE))
if set(FRAME_CORE) - set(P.LADDER_RUNGS):
    raise AssertionError(
        "cell 9b names %s, which is not on the ladder. The frame arm is a "
        "control ON the ladder, and cell 9c reads its rungs into one table."
        % ", ".join(sorted(set(FRAME_CORE) - set(P.LADDER_RUNGS))))

# The same naming 9b uses, so the resume logic and cell 9c find these files
# without being told about them a second time.
FRAME_OUT.update({g: RUNS / ("ex2_q3_dims_%s_%s.jsonl" % (g, FRAME_ALT))
                  for g in FRAME_REST})
FRAME_BASE.update({g: rung_file("dims", g) for g in FRAME_REST})

# WIDENED FOR CELL 9c, whether or not this cell ends up spending. 9c loops
# over FRAME_RUNGS and skips a rung whose file is not on disk, so the
# read-out covers exactly what has been bought. Unconditional because a
# widening that depended on the spend gate would make the read-out's shape
# depend on the order the cells were run in.
FRAME_RUNGS = tuple(g for g in P.LADDER_RUNGS if g in FRAME_OUT)

# THE EDIT IS CHECKED AT EVERY RUNG, not only the four bought here. The four
# are read against 9b's three in one table, so the manipulation has to be
# the same at all seven or the frame and the instruction cannot be told
# apart -- which is the whole claim the control makes.
import difflib
print("=" * 70)
print("THE FRAME EDIT: %s minus named, at every rung of the ladder" % FRAME_ALT)
print("=" * 70)
_edits = {}
for g in P.LADDER_RUNGS:
    _a = P.system_prompt(g, "dims", dims_frame="named").splitlines()
    _b = P.system_prompt(g, "dims", dims_frame=FRAME_ALT).splitlines()
    _d = list(difflib.unified_diff(_a, _b, lineterm="", n=0))
    _edits[g] = (
        tuple(l[1:] for l in _d
              if l.startswith("+") and not l.startswith("+++")),
        tuple(l[1:] for l in _d
              if l.startswith("-") and not l.startswith("---")))
if len(set(_edits.values())) != 1:
    raise AssertionError(
        "the %s edit is not the same at every rung (%s differ). The frame "
        "contrast has to be one manipulation everywhere, or a difference "
        "between rungs is the wording and not the instruction."
        % (FRAME_ALT, ", ".join(sorted(g for g in P.LADDER_RUNGS
                                       if _edits[g] != _edits["N0"]))))
_add, _rem = _edits[FRAME_REST[0]]
for l in _rem:
    if l.strip():
        print("   - " + l)
for l in _add:
    if l.strip():
        print("   + " + l)
print()
print("PASS  one edit, identical at all %d rungs, so the %d bought here are"
      % (len(P.LADDER_RUNGS), len(FRAME_REST)))
print("      the same control cell 9b ran and belong in its table.")

# THE PROMPT SENT, at the gate rung. 9b prints N0's; this prints the rung
# the gate implicated, which is the one whose frame arm the chapter will be
# asked about first.
_show = GATE_RUNG if GATE_RUNG in FRAME_REST else FRAME_REST[0]
print()
print("=" * 70)
print("THE FULL PROMPT SENT, dims %s under %s" % (_show, FRAME_ALT))
print("=" * 70)
print(P.system_prompt(_show, "dims", dims_frame=FRAME_ALT))
print("=" * 70)

# ONE REPEAT COUNT FOR THE WHOLE ARM, read from disk exactly as 9b reads it.
# Checked across 9b's rungs as well as these four: if the four came in at a
# different count, the frame column in cell 9c would be three rungs at one
# precision and four at another and the delta column would not be one
# contrast.
_named = {g: repeats_on_disk(FRAME_BASE[g], g) for g in FRAME_RUNGS}
if 0 in _named.values():
    print()
    print("The named-frame rungs this is read against are not all on disk:")
    for g in FRAME_RUNGS:
        print("  %-8s %-44s %s" % (g, rel(FRAME_BASE[g]),
                                   "%d repeats" % _named[g] if _named[g]
                                   else "MISSING"))
    print("Cells 9 and 9d buy them. Nothing to control against, nothing bought.")
elif len(set(_named.values())) != 1:
    raise AssertionError(
        "the named rungs carry different repeat counts (%s). The frame arm "
        "can only match one of them, and a control matched to some rungs and "
        "not others cannot be read as one contrast." % _named)
else:
    FRAME_REPEATS = next(iter(_named.values()))
    n_calls = (len(CALL_SCENES) * len(MODELS) * FRAME_REPEATS
               * len(FRAME_REST))
    print()
    print("COST: %d scenes x %d models x %d repeats x %d rungs = %d calls"
          % (len(CALL_SCENES), len(MODELS), FRAME_REPEATS, len(FRAME_REST),
             n_calls))
    print("      rungs: %s" % ", ".join(FRAME_REST))
    print("      repeats read from disk, matched to the named ladder and to")
    print("      the three rungs cell 9b bought.")
    print()
    _per = len(CALL_SCENES) * len(MODELS) * FRAME_REPEATS
    for g in FRAME_REST:
        print("      %-8s %-40s %4d of %d answered"
              % (g, rel(FRAME_OUT[g]), answered(FRAME_OUT[g]), _per))
    print()
    print("      CELL 9b's, on disk already and costing nothing here:")
    for g in FRAME_CORE:
        print("      %-8s %-40s %4d of %d answered"
              % (g, rel(FRAME_OUT[g]), answered(FRAME_OUT[g]), _per))
    print()
    print("      solo appends the frame to the trial_id when it is not the")
    print("      default, so these ids cannot collide with the named runs'.")
    print("      A part-run cell resumes: only the missing repeats are")
    print("      bought, and an unparseable reply is retried rather than")
    print("      skipped.")
    print()
    print("      EVERY MODEL, for the reason 9b gives: gemini reads the pose")
    print("      at ceiling under the named frame, so a frame effect in")
    print("      another model means nothing unless gemini is in the same")
    print("      run and stays put.")

    CONFIRM_SPEND = None        # <-- set to the number in the COST line

    if spend_gate(n_calls, CONFIRM_SPEND,
                  factors=(("scenes", len(CALL_SCENES)),
                           ("models", len(MODELS)),
                           ("repeats", FRAME_REPEATS),
                           ("rungs", len(FRAME_REST)))):
        for g in FRAME_REST:
            print("\n--- dims %s, %s frame ---" % (g, FRAME_ALT))
            S.run(str(CAPTURES), out_path=str(FRAME_OUT[g]), models=MODELS,
                  conditions=("dims",), preferences=(PREFERENCE,),
                  rungs=(g,), modalities=("V",), kind="pair",
                  repeats=FRAME_REPEATS, dims_frames=(FRAME_ALT,))
        print("\nthe %s arm now covers every rung of the ladder" % FRAME_ALT)

THE FRAME EDIT: extents minus named, at every rung of the ladder
   - Each object states its mass, "mass_kg", whether it is "delicate", its height,
   - width and depth measured standing on its smallest face, "size_upright_m", and
   - the arms that can reach it, "arms_that_can_reach". No opening and no resting
   - face are given.
   + Each object states its mass, "mass_kg", whether it is "delicate", its three
   + extents largest first, "extents_m", and the arms that can reach it,
   + "arms_that_can_reach". No opening and no resting face are given.

PASS  one edit, identical at all 7 rungs, so the 4 bought here are
      the same control cell 9b ran and belong in its table.

THE FULL PROMPT SENT, dims N-D under extents
You are the task allocator for a four-arm robotic cell. Each time you are
asked, choose ONE queued task and ONE idle arm, or choose to wait.

THE CELL
A 2.8 x 1.6 m table, origin at its centre, x east, y north. Two UR10 arms sit
mid-table on the west and east edges, t


COST: 68 scenes x 3 models x 3 repeats x 4 rungs = 2448 calls
      rungs: N-A, N-C, N-order, N-D
      repeats read from disk, matched to the named ladder and to
      the three rungs cell 9b bought.

      N-A      runs/ex2_q3_dims_N-A_extents.jsonl          0 of 612 answered
      N-C      runs/ex2_q3_dims_N-C_extents.jsonl          0 of 612 answered
      N-order  runs/ex2_q3_dims_N-order_extents.jsonl      0 of 612 answered
      N-D      runs/ex2_q3_dims_N-D_extents.jsonl          0 of 612 answered

      CELL 9b's, on disk already and costing nothing here:
      N0       runs/ex2_q3_dims_N0_extents.jsonl         612 of 612 answered


      N-CD     runs/ex2_q3_dims_N-CD_extents.jsonl       612 of 612 answered
      N-ACD    runs/ex2_q3_dims_N-ACD_extents.jsonl      612 of 612 answered

      solo appends the frame to the trial_id when it is not the
      default, so these ids cannot collide with the named runs'.
      A part-run cell resumes: only the missing repeats are
      bought, and an unparseable reply is retried rather than
      skipped.

      EVERY MODEL, for the reason 9b gives: gemini reads the pose
      at ceiling under the named frame, so a frame effect in
      another model means nothing unless gemini is in the same
      run and stays put.
set CONFIRM_SPEND = 2448 in this cell to proceed

not confirmed; no calls made.


## Cell 9c. Frame read-out

No model calls. The contrast says where each model lands. The **anchor** says
where its answer sat: how often it named `small_face`, and how often it reported
the 0.050 m opening that follows from standing on the smallest face.

Read the anchor, not the accuracy. The two faces are balanced, so a model that
always answers `small_face` scores 50% correct and reads as a coin flip. What
separates seeing the pose from reading the wording is whether the anchor **moves
when the frame moves**.

`N0` uses the base schema and does not report `resting_face`, so its anchor is
the reported opening alone.

In [14]:
# --- Cell 9c. Frame read-out. No model calls. -------------------------------
TOL = 0.006                # grade.classify_width's tolerance, not a new one
SMALL_OPENING = 0.050      # the opening for a block on its SMALLEST face,
                           # which is the orientation the named frame states

def frame_rows(path, rung, model):
    if not pathlib.Path(path).exists():
        return []
    rows, _ = load_run(path, "dims", MODELS)
    rows = [r for r in rows if r.get("rung") == rung]
    return [r for r in keep_analysable(rows, USABLE) if r["model"] == model]

_have = [g for g in FRAME_RUNGS if pathlib.Path(FRAME_OUT[g]).exists()]
if not _have:
    print("Cell 9b has not been run, so there is no %s arm to read." % FRAME_ALT)
    print("Nothing here is a null; there is simply no control yet.")
else:
    contrast_out, anchor_out = [], []
    for g in _have:
        for model in MODELS:
            a = paired_diffs(frame_rows(FRAME_OUT[g], g, model), USABLE,
                             "small_face", "large_face")
            b = paired_diffs(frame_rows(FRAME_BASE[g], g, model), USABLE,
                             "small_face", "large_face")
            m_a, _, _, n_a = paired_mean_ci([d for _, d in a])
            m_b, _, _, n_b = paired_mean_ci([d for _, d in b])
            mean, lo, hi, npos = paired_mean_ci(
                [d for _, d in paired_delta(a, b)])
            if not npos:
                v = "NOT RUN"
            elif lo > 0 or hi < 0:
                v = "FRAME EFFECT"
            elif hi < GATE_MIN and lo > -GATE_MIN:
                v = "no frame effect"
            else:
                v = "UNRESOLVED"
            contrast_out.append([g, model, n_b, fmt(m_b), n_a, fmt(m_a),
                                 fmt(mean), fmt(lo), fmt(hi), v])

            for label, path in (("named", FRAME_BASE[g]),
                                (FRAME_ALT, FRAME_OUT[g])):
                rs = frame_rows(path, g, model)
                named = [r for r in rs if r.get("resting_face")]
                opened = [r for r in rs
                          if r.get("opening_needed_m") is not None]
                anchor_out.append([
                    g, model, label, len(rs),
                    fmt(pct(sum(1 for r in opened
                                if abs(r["opening_needed_m"] - SMALL_OPENING)
                                < TOL), len(opened))),
                    fmt(pct(sum(1 for r in named
                                if r["resting_face"] == "small_face"),
                            len(named))) if named else "not asked",
                    fmt(pct(sum(1 for r in named
                                if r["resting_face"] == r.get("true_pose")),
                            len(named))) if named else "not asked"])

    show(["rung", "model", "n", "named", "n", FRAME_ALT,
          "delta", "lo", "hi", "verdict"], contrast_out)
    print()
    print("delta is %s minus named, paired within position at both frames."
          % FRAME_ALT)
    print()
    show(["rung", "model", "frame", "n", "reports 0.050", "says small_face",
          "face correct"], anchor_out)
    write_csv("tab_ex2_q3_frame.csv",
              ["rung", "model", "frame", "n_trials",
               "reports_small_opening_pct", "says_small_face_pct",
               "face_correct_pct"], anchor_out)
    print()
    print("=" * 70)
    print("WHAT THE TWO OUTCOMES MEAN, stated before the numbers are read")
    print("=" * 70)
    print("  anchor STAYS PUT   step 1 was never the problem. The model")
    print("                     already treated the stated size as a")
    print("                     convention, and the dims ladder measured")
    print("                     what it claims to measure.")
    print()
    print("  anchor FOLLOWS     the answer tracked the wording rather than")
    print("  the frame          the picture, and every dims number for that")
    print("                     model is about the glossary, not grounding.")
    print()
    print("GEMINI IS THE INTERNAL CONTROL. It reads the pose at 100% under")
    print("named. If its anchor moves too, the frame changed something other")
    print("than step 1 and neither column is about grounding for any model.")
    print()
    print("READ N0 FIRST. If a model already scores well there under %s," % FRAME_ALT)
    print("the instruction ladder was compensating for the glossary rather")
    print("than for a failure to ground, and cell 12's contrasts are about")
    print("the prompt's own wording.")

rung   model      n   named  n   extents  delta  lo    hi    verdict        
-----  ---------  --  -----  --  -------  -----  ----  ----  ---------------
N0     gpt_hi     32  2.1    32  7.3      5.2    -4.6  15.0  no frame effect
N0     gemini     32  46.9   32  94.8     47.9   35.5  60.3  FRAME EFFECT   
N0     claude_md  32  -10.4  32  -1.0     9.4    -5.1  23.8  no frame effect
N-CD   gpt_hi     32  32.3   32  76.0     43.8   31.9  55.6  FRAME EFFECT   
N-CD   gemini     32  100.0  32  100.0    0.0    0.0   0.0   no frame effect
N-CD   claude_md  32  0.0    32  -2.1     -2.1   -7.9  3.7   no frame effect
N-ACD  gpt_hi     32  53.1   32  88.5     35.4   24.5  46.4  FRAME EFFECT   
N-ACD  gemini     32  100.0  32  100.0    0.0    0.0   0.0   no frame effect
N-ACD  claude_md  32  0.0    32  2.1      2.1    -0.8  4.9   no frame effect

delta is extents minus named, paired within position at both frames.

rung   model      frame    n    reports 0.050  says small_face  face correct
-----

wrote tables/ex2_q3/tab_ex2_q3_frame.csv  (18 rows)

WHAT THE TWO OUTCOMES MEAN, stated before the numbers are read
  anchor STAYS PUT   step 1 was never the problem. The model
                     already treated the stated size as a
                     convention, and the dims ladder measured
                     what it claims to measure.

  anchor FOLLOWS     the answer tracked the wording rather than
  the frame          the picture, and every dims number for that
                     model is about the glossary, not grounding.

GEMINI IS THE INTERNAL CONTROL. It reads the pose at 100% under
named. If its anchor moves too, the frame changed something other
than step 1 and neither column is about grounding for any model.

READ N0 FIRST. If a model already scores well there under extents,
the instruction ladder was compensating for the glossary rather
than for a failure to ground, and cell 12's contrasts are about
the prompt's own wording.


## Cell 9d. The ceiling: the complete procedure in `dims`

**Makes model calls.** The ladder tests each factor alone and one pair. It never
tests the whole procedure. `N-ACD` is `A + C + D` together: look at the image,
here is how the opening follows from the resting face, and state the face and
the opening before you name an arm.

**Why it is needed.** Without it a model that stops short at `N-CD` cannot be
told apart from one that was never asked for every step. *Cannot derive* and
*was not instructed to derive* predict the same number. `N-ACD` bounds what
instruction can achieve, which is what EX1's `givenset` does for that
experiment's spine.

**It is not pre-registered.** The first six rungs are; this was added on
2026-09-01 after they had been run, and `prompts.PRE_REGISTERED_LADDER` is what
any table presenting the ladder as a pre-registered design must read.

`A_ATTEND` is used unchanged, so `N-ACD` minus `N-CD` is the attention sentence
and nothing else. A stronger sentence naming the resting face as visible would
be a new factor with no single-factor cell of its own, and it would cross into
telling the model where the answer is, which is the directives' job.

In [15]:
# --- Cell 9d. CEILING: dims N-ACD. MAKES MODEL CALLS. -----------------------
CEIL_RUNG = "N-ACD"
CEIL_OUT = rung_file("dims", CEIL_RUNG)
CEIL_BASE = rung_file("dims", "N-CD")       # the rung it is read against

# The ceiling is only interpretable as A on top of C and D. If the rung ever
# stops being exactly that, the contrast stops being the attention sentence
# and the cell measures something it does not claim to.
if P.RUNGS[CEIL_RUNG]["text"] != P.A_ATTEND + P.C_DERIVE + P.D_ELICIT:
    raise AssertionError(
        "%s is not A_ATTEND + C_DERIVE + D_ELICIT. Its whole attribution is "
        "that it is N-CD plus the attention sentence, so %s minus N-CD is "
        "that sentence and nothing else." % (CEIL_RUNG, CEIL_RUNG))
if CEIL_RUNG in P.DIRECTIVE_RUNGS:
    raise AssertionError(
        "%s is classified as a directive. Every table that splits the ladder "
        "from the directives would file the ceiling with X-image, which "
        "presents a scaffolding rung as a provenance instruction."
        % CEIL_RUNG)
if CEIL_RUNG in P.PRE_REGISTERED_LADDER:
    raise AssertionError(
        "%s is inside PRE_REGISTERED_LADDER. It was added after the ladder "
        "was run and must not be presented as pre-registered." % CEIL_RUNG)

if not pathlib.Path(CEIL_BASE).exists():
    print("N-CD in dims is not on disk:")
    print("  %s" % rel(CEIL_BASE))
    print("Cell 9 buys it. The ceiling is read against it, so nothing bought.")
else:
    n_calls = len(CALL_SCENES) * len(MODELS) * REPEATS
    print("COST: %d scenes x %d models x %d repeat = %d calls"
          % (len(CALL_SCENES), len(MODELS), REPEATS, n_calls))
    print("      dims %s, one cell." % CEIL_RUNG)
    print("      %3d of %d already answered" % (answered(CEIL_OUT), n_calls))
    print()
    print("      READ AGAINST, already on disk and costing nothing:")
    for _r in ("N0", "N-CD"):
        _f = N0_FILE["dims"] if _r == "N0" else rung_file("dims", _r)
        print("        %-5s %-44s %3d answered"
              % (_r, rel(_f), answered(_f) if pathlib.Path(_f).exists() else 0))
    print()
    print("      dims ONLY. In conflict_face the same rung would confound the")
    print("      procedure with precedence, which is what X-image measures.")
    print("      Keeping them in separate questions keeps both attributable.")

    CONFIRM_SPEND = None        # <-- set to the number in the COST line

    if spend_gate(n_calls, CONFIRM_SPEND, CEIL_OUT,
                  factors=(("scenes", len(CALL_SCENES)),
                           ("models", len(MODELS)), ("repeats", REPEATS))):
        S.run(str(CAPTURES), out_path=str(CEIL_OUT), models=MODELS,
              conditions=("dims",), preferences=(PREFERENCE,),
              rungs=(CEIL_RUNG,), modalities=("V",), kind="pair",
              repeats=REPEATS)
        print("answered now:", answered(CEIL_OUT))

COST: 68 scenes x 3 models x 3 repeat = 612 calls
      dims N-ACD, one cell.
      612 of 612 already answered

      READ AGAINST, already on disk and costing nothing:
        N0    runs/ex2_q1_dims_N0.jsonl                    612 answered
        N-CD  runs/ex2_q3_dims_N-CD.jsonl                  612 answered

      dims ONLY. In conflict_face the same rung would confound the
      procedure with precedence, which is what X-image measures.
      Keeping them in separate questions keeps both attributable.
already answered: 612 of 612 in ex2_q3_dims_N-ACD.jsonl
set CONFIRM_SPEND = 612 in this cell to proceed

not confirmed; no calls made.


## Cell 9e. Ceiling read-out

No model calls. Three readings, and the third is the one the cell exists for.

The **contrast** is where instruction tops out. The **delta against `N-CD`** is
what the attention sentence adds once derivation and elicitation are already in
place. The **face accuracy** says whether a model that still fails is failing to
see the pose or failing to act on the pose it named, which the arm contrast on
its own cannot distinguish.

In [16]:
# --- Cell 9e. Ceiling read-out. No model calls. -----------------------------
def ceil_rows(path, rung, model):
    if not pathlib.Path(path).exists():
        return []
    rows, _ = load_run(path, "dims", MODELS)
    rows = [r for r in rows if r.get("rung") == rung]
    return [r for r in keep_analysable(rows, USABLE) if r["model"] == model]

if not pathlib.Path(CEIL_OUT).exists():
    print("Cell 9d has not been run, so there is no ceiling to read.")
    print("This is not a null. There is simply no bound yet on what")
    print("instruction can achieve, and N-CD remains the highest rung.")
else:
    ceil_out, face_out = [], []
    for model in MODELS:
        a = paired_diffs(ceil_rows(CEIL_OUT, CEIL_RUNG, model), USABLE,
                         "small_face", "large_face")
        m_a, alo, ahi, n_a = paired_mean_ci([d for _, d in a])
        row = [model, fmt(m_a), fmt(alo), fmt(ahi), n_a]
        for base_rung, base_path in (("N-CD", CEIL_BASE),
                                     ("N0", N0_FILE["dims"])):
            b = paired_diffs(ceil_rows(base_path, base_rung, model), USABLE,
                             "small_face", "large_face")
            if not any(d is not None for _, d in b):
                row += ["NA", "NA", "NA"]
                continue
            mean, lo, hi, npos = paired_mean_ci(
                [d for _, d in paired_delta(a, b)])
            row += [fmt(mean), fmt(lo), fmt(hi)]
        ceil_out.append(row)

        # Did it name the face, and did it name it right? Only asked under
        # the face-first schema, which both N-ACD and N-CD use.
        for label, path, rung in ((CEIL_RUNG, CEIL_OUT, CEIL_RUNG),
                                  ("N-CD", CEIL_BASE, "N-CD")):
            rs = ceil_rows(path, rung, model)
            named = [r for r in rs if r.get("resting_face")]
            right = sum(1 for r in named
                        if r["resting_face"] == r.get("true_pose"))
            face_out.append([model, label, len(rs), len(named),
                             fmt(pct(right, len(named)))])

    show(["model", CEIL_RUNG, "lo", "hi", "npos",
          "vs N-CD", "lo", "hi", "vs N0", "lo", "hi"], ceil_out)
    print()
    print("vs N-CD is the ATTENTION SENTENCE, on top of derivation and")
    print("elicitation. vs N0 is everything instruction bought in dims.")
    print()
    show(["model", "rung", "n", "named", "face correct %"], face_out)
    write_csv("tab_ex2_q3_ceiling.csv",
              ["model", "rung", "n_trials", "n_named", "face_correct_pct"],
              face_out)
    print()
    print("=" * 70)
    print("HOW TO READ THE CEILING, stated before the numbers")
    print("=" * 70)
    print("A model still short of +100 here was told every step and did not")
    print("complete them. That bounds INSTRUCTION, not the model: a rung")
    print("cannot rule out that a differently worded procedure would do")
    print("better, and the chapter must say so rather than reading this as")
    print("a capability limit.")
    print()
    print("The face column separates the two ways of falling short:")
    print("  face right, contrast low   -> it saw the pose and did not act")
    print("                                on the pose it named")
    print("  face wrong, contrast low   -> it never had the pose to act on,")
    print("                                and no wording about DERIVING")
    print("                                from the pose can repair that")
    print()
    print("At %d positions and %d repeat an interval here is about %.0f points"
          % (len(USABLE), REPEATS, GATE_MIN))
    print("wide whatever the data, so read a small delta as unresolved.")

model      N-ACD  lo     hi     npos  vs N-CD  lo   hi    vs N0  lo    hi  
---------  -----  -----  -----  ----  -------  ---  ----  -----  ----  ----
gpt_hi     53.1   42.6   63.6   32    20.8     7.8  33.9  51.0   35.8  66.3
gemini     100.0  100.0  100.0  32    0.0      0.0  0.0   53.1   40.7  65.5
claude_md  0.0    0.0    0.0    32    0.0      0.0  0.0   10.4   -3.5  24.3

vs N-CD is the ATTENTION SENTENCE, on top of derivation and
elicitation. vs N0 is everything instruction bought in dims.

model      rung   n    named  face correct %
---------  -----  ---  -----  --------------
gpt_hi     N-ACD  192  192    76.6          
gpt_hi     N-CD   192  192    66.1          
gemini     N-ACD  192  192    100.0         
gemini     N-CD   192  192    100.0         
claude_md  N-ACD  192  192    50.0          
claude_md  N-CD   192  192    50.5          
wrote tables/ex2_q3/tab_ex2_q3_ceiling.csv  (6 rows)

HOW TO READ THE CEILING, stated before the numbers
A model still short of +100 here

## Cell 10. The conflict-face ladder

**Makes model calls.** Every rung except `N0`, for every model, one repeat.

**Why `conflict_face` and not `conflict`.** Q3 asks which instruction moves a
model from following the text to using the scene. In `conflict` the state
supplies `opening_needed_m` and R3 names that field, so following the text is
**rule-compliant** and a rung effect there would answer a different question:
does the instruction make the model override a supplied value. In
`conflict_face` the number is withheld, R3 names nothing, and the only route to
an opening is a face -- one asserted by the text, one visible in the image. A
rung effect there is remediation of precedence, which is what Q3 asks about.

The baseline is Q2's `conflict_face` at N0, where GPT reads -89.6 and Gemini
-100.0 against ceilings of +87.5 and +100.0. Claude is a null in both and has
no source to prefer, so a rung that moves it would be teaching it to derive at
all rather than to prefer the scene.

In [17]:
# --- Cell 10. The conflict_face ladder. MAKES MODEL CALLS. ------------------
CONFLICT_RUNGS = tuple(r for r in LADDER if r != "N0")

if not pathlib.Path(N0_FILE["conflict_face"]).exists():
    print("Q2's conflict_face N0 file is not on disk:")
    print("  %s" % rel(N0_FILE["conflict_face"]))
    print("Every contrast here is against it, so this cannot be read until")
    print("Q2 cell 6b has been run. Nothing bought.")
else:
    n_calls = len(CALL_SCENES) * len(MODELS) * REPEATS * len(CONFLICT_RUNGS)
    print("COST: %d scenes x %d models x %d repeat x %d rungs = %d calls"
          % (len(CALL_SCENES), len(MODELS), REPEATS, len(CONFLICT_RUNGS),
             n_calls))
    print("      rungs: %s" % ", ".join(CONFLICT_RUNGS))
    for rung in CONFLICT_RUNGS:
        _f = rung_file("conflict_face", rung)
        print("      %-8s %3d of %d answered"
              % (rung, answered(_f), len(CALL_SCENES) * len(MODELS)))

    CONFIRM_SPEND = None        # <-- set to the number in the COST line

    if spend_gate(n_calls, CONFIRM_SPEND,
                  factors=(("scenes", len(CALL_SCENES)),
                           ("models", len(MODELS)), ("repeats", REPEATS),
                           ("rungs", len(CONFLICT_RUNGS)))):
        for rung in CONFLICT_RUNGS:
            out = rung_file("conflict_face", rung)
            print("\n--- conflict_face %s ---" % rung)
            S.run(str(CAPTURES), out_path=str(out), models=MODELS,
                  conditions=("conflict_face",), preferences=(PREFERENCE,),
                  rungs=(rung,), modalities=("V",), kind="pair",
                  repeats=REPEATS)
        print("\nconflict_face ladder complete")

COST: 68 scenes x 3 models x 3 repeat x 5 rungs = 3060 calls
      rungs: N-A, N-C, N-order, N-D, N-CD
      N-A      612 of 204 answered
      N-C      612 of 204 answered
      N-order  612 of 204 answered
      N-D      612 of 204 answered
      N-CD     612 of 204 answered
set CONFIRM_SPEND = 3060 in this cell to proceed

not confirmed; no calls made.


## Cell 10b. Stage 1: the precedence directive in `conflict_face`

**Makes model calls.** The run this notebook exists to add: does telling a model
*where the image and the stated resting face disagree, go by the image* move it
off the false stated face?

Read against two files already on disk. `N0` is the baseline. `N-A` is the more
informative one: it says "look at the image" without naming the face or admitting
a conflict, and `X-image` is literally that text plus one sentence, so
`X-image - N-A` is the precedence sentence and nothing else.

In [18]:
# --- Cell 10b. STAGE 1: X-image in conflict_face. MAKES MODEL CALLS. --------
DIRECTIVE_COND, DIRECTIVE = DIRECTIVE_STAGE1[0]
DIRECTIVE_OUT = rung_file(DIRECTIVE_COND, DIRECTIVE)

if not pathlib.Path(N0_FILE[DIRECTIVE_COND]).exists():
    print("Q2's %s N0 file is not on disk:" % DIRECTIVE_COND)
    print("  %s" % rel(N0_FILE[DIRECTIVE_COND]))
    print("The headline contrast is against it, so this cannot be read until")
    print("Q2 has been run. Nothing bought.")
else:
    n_calls = len(CALL_SCENES) * len(MODELS) * REPEATS
    print("COST: %d scenes x %d models x %d repeats = %d calls"
          % (len(CALL_SCENES), len(MODELS), REPEATS, n_calls))
    print("      %s in %s, one cell." % (DIRECTIVE, DIRECTIVE_COND))
    print("      %3d of %d already answered"
          % (answered(DIRECTIVE_OUT), n_calls))
    print()
    print("      READ AGAINST, already on disk and costing nothing:")
    for _r in ("N0", "N-A"):
        _f = N0_FILE[DIRECTIVE_COND] if _r == "N0" else rung_file(
            DIRECTIVE_COND, _r)
        print("        %-4s %-46s %3d answered"
              % (_r, rel(_f), answered(_f) if pathlib.Path(_f).exists() else 0))
    print()
    print("      dims is NOT here and cannot be: it states no resting face,")
    print("      so the directive is vacuous there and solo refuses to render")
    print("      it before a single call is made.")

    CONFIRM_SPEND = None        # <-- set to the number in the COST line

    if spend_gate(n_calls, CONFIRM_SPEND, DIRECTIVE_OUT,
                  factors=(("scenes", len(CALL_SCENES)),
                           ("models", len(MODELS)), ("repeats", REPEATS))):
        S.run(str(CAPTURES), out_path=str(DIRECTIVE_OUT), models=MODELS,
              conditions=(DIRECTIVE_COND,), preferences=(PREFERENCE,),
              rungs=(DIRECTIVE,), modalities=("V",), kind="pair",
              repeats=REPEATS)
        print("answered now:", answered(DIRECTIVE_OUT))

COST: 68 scenes x 3 models x 3 repeats = 612 calls
      X-image in conflict_face, one cell.
      408 of 612 already answered

      READ AGAINST, already on disk and costing nothing:
        N0   runs/ex2_q2_conflict_face_N0.jsonl             612 answered
        N-A  runs/ex2_q3_conflict_face_N-A.jsonl            612 answered

      dims is NOT here and cannot be: it states no resting face,
      so the directive is vacuous there and solo refuses to render
      it before a single call is made.
already answered: 408 of 612 in ex2_q3_conflict_face_X-image.jsonl
set CONFIRM_SPEND = 612 in this cell to proceed

not confirmed; no calls made.


## Cell 10c. Stage 1 read-out, and the gate on stage 2

No model calls. Three outcomes per model, as in cell 7: **moved**, **not
moved**, **unresolved**. The two controls in cell 10d exist only to interpret a
model that moved, so a model that did not move buys nothing further.

In [19]:
# --- Cell 10c. Directive read-out. No model calls. --------------------------
DIRECTIVE_VERDICT = {}
dir_read = []
for model in MODELS:
    a = contrast_pairs(DIRECTIVE_COND, DIRECTIVE, model)
    b = contrast_pairs(DIRECTIVE_COND, "N0", model)
    c = contrast_pairs(DIRECTIVE_COND, "N-A", model)
    m_0, _, _, _ = paired_mean_ci([d for _, d in b])
    m_a, _, _, _ = paired_mean_ci([d for _, d in c])
    m_x, _, _, _ = paired_mean_ci([d for _, d in a])
    mean, lo, hi, npos = paired_mean_ci([d for _, d in paired_delta(a, b)])
    _sent = paired_mean_ci([d for _, d in paired_delta(a, c)])
    if not npos:
        v = "NOT RUN"
    elif lo > 0:
        v = "MOVED"
    elif hi < 0:
        v = "MOVED BACKWARDS"
    elif hi < GATE_MIN:
        v = "NOT MOVED"
    else:
        v = "UNRESOLVED"
    DIRECTIVE_VERDICT[model] = v
    dir_read.append([model, fmt(m_0), fmt(m_a), fmt(m_x), npos, fmt(mean),
                     fmt(lo), fmt(hi), fmt(_sent[0]), v])

show(["model", "N0", "N-A", DIRECTIVE, "npos", "vs N0", "lo", "hi",
      "vs N-A", "verdict"], dir_read)
write_csv("tab_ex2_q3_directive_gate.csv",
          ["model", "n0", "n_a", "directive", "npos", "delta_vs_n0",
           "lo", "hi", "delta_vs_n_a", "verdict"], dir_read)
print()
print("\"vs N0\" is the total movement a direct instruction buys. \"vs N-A\"")
print("subtracts being told to look at the image, leaving the precedence")
print("sentence alone. Both are paired within position at each level.")
print()
print("A model reads UNRESOLVED when the interval still admits an effect of")
print("%.0f points or more. That is not a null." % GATE_MIN)
print()
DIRECTIVE_PROCEED = [m for m in MODELS
                     if DIRECTIVE_VERDICT[m] in ("MOVED", "MOVED BACKWARDS",
                                                 "UNRESOLVED")]
for model in MODELS:
    print("  %-9s %s" % (model, DIRECTIVE_VERDICT[model]))
if DIRECTIVE_PROCEED:
    print()
    print("STAGE 2 IS WORTH BUYING. %s did not come back a flat null, and a"
          % ", ".join(DIRECTIVE_PROCEED))
    print("number that moved is exactly the number the two controls are")
    print("needed to interpret:")
    print("  congruent_face @ X-image  rules out that ANY extra sentence")
    print("                            would have done it")
    print("  conflict_face  @ X-state  rules out that the model merely heard")
    print("                            the word \"image\"")
else:
    print()
    print("STAGE 2 BUYS NOTHING. No model moved, so there is no effect for")
    print("either control to explain away. Report the null and stop; cell 10d")
    print("will refuse on its own gate.")

model      N0      N-A    X-image  npos  vs N0  lo     hi     vs N-A  verdict  
---------  ------  -----  -------  ----  -----  -----  -----  ------  ---------
gpt_hi     -89.6   -80.2  -1.6     32    88.0   72.3   103.8  78.6    MOVED    
gemini     -100.0  -96.9  100.0    32    200.0  200.0  200.0  196.9   MOVED    
claude_md  7.3     -4.2   -15.6    32    -22.9  -46.5  0.7    -11.5   NOT MOVED
wrote tables/ex2_q3/tab_ex2_q3_directive_gate.csv  (3 rows)

"vs N0" is the total movement a direct instruction buys. "vs N-A"
subtracts being told to look at the image, leaving the precedence
sentence alone. Both are paired within position at each level.

A model reads UNRESOLVED when the interval still admits an effect of
30 points or more. That is not a null.

  gpt_hi    MOVED
  gemini    MOVED
  claude_md NOT MOVED

STAGE 2 IS WORTH BUYING. gpt_hi, gemini did not come back a flat null, and a
number that moved is exactly the number the two controls are
needed to interpret:
  congruent_face

## Cell 10d. Stage 2: the two controls

**Makes model calls.** Bought only if cell 10c says a model moved. Each control
rules out one competing explanation for that movement, and neither is worth
buying against a flat null.

In [20]:
# --- Cell 10d. STAGE 2: the directive controls. MAKES MODEL CALLS. ----------
if not DIRECTIVE_PROCEED:
    print("Cell 10c reports no movement in any model, so neither control has")
    print("anything to control for. Nothing bought.")
else:
    n_calls = len(CALL_SCENES) * len(MODELS) * REPEATS * len(DIRECTIVE_STAGE2)
    print("COST: %d scenes x %d models x %d repeats x %d cells = %d calls"
          % (len(CALL_SCENES), len(MODELS), REPEATS, len(DIRECTIVE_STAGE2),
             n_calls))
    for _c, _r in DIRECTIVE_STAGE2:
        print("      %-15s %-8s %3d of %d answered"
              % (_c, _r, answered(rung_file(_c, _r)),
                 len(CALL_SCENES) * len(MODELS) * REPEATS))

    CONFIRM_SPEND = None        # <-- set to the number in the COST line

    if spend_gate(n_calls, CONFIRM_SPEND,
                  factors=(("scenes", len(CALL_SCENES)),
                           ("models", len(MODELS)), ("repeats", REPEATS),
                           ("cells", len(DIRECTIVE_STAGE2)))):
        for _c, _r in DIRECTIVE_STAGE2:
            out = rung_file(_c, _r)
            print("\n--- %s %s ---" % (_c, _r))
            S.run(str(CAPTURES), out_path=str(out), models=MODELS,
                  conditions=(_c,), preferences=(PREFERENCE,), rungs=(_r,),
                  modalities=("V",), kind="pair", repeats=REPEATS)
        print("\ndirective controls complete")

COST: 68 scenes x 3 models x 3 repeats x 2 cells = 1224 calls


      congruent_face  X-image  408 of 612 answered


      conflict_face   X-state  408 of 612 answered
set CONFIRM_SPEND = 1224 in this cell to proceed

not confirmed; no calls made.


## Cell 11. What is on disk

No model calls. One row per condition and rung, with the row count, the models
present and whether the file's own `rung` field matches the file it is in.

That last check is not ceremony: a rung recorded under the wrong name is
exactly the failure `solo`'s `trial_id` scheme exists to prevent, and it would
show up as a rung that mysteriously did nothing.

In [21]:
# --- Cell 11. Rung inventory. No model calls. -------------------------------
inv, problems = [], []
AVAILABLE = []
# The directive cells go in as their own (condition, rung) entries rather
# than being folded into LADDER: conflict_face then appears twice, once for
# the ladder and once for its directives, and congruent_face gains a rung
# without becoming a ladder condition. Everything downstream reads
# AVAILABLE, so cells 12 and 14 pick them up for free.
_plan = ([(c, LADDER) for c in CONDITIONS]
         + [(c, ("N0",)) for c in BASELINES]
         + [(c, (r,)) for c, r in DIRECTIVE_CELLS])
for cond, _rungs in _plan:
    for rung in _rungs:
        path = N0_FILE[cond] if rung == "N0" else rung_file(cond, rung)
        if not pathlib.Path(path).exists():
            inv.append([cond, rung, rel(path), 0, "", "-", "missing"])
            continue
        rows, _ = load_run(path, cond, MODELS)
        wrong = sorted({r.get("rung") for r in rows} - {rung})
        if wrong:
            problems.append("%s holds rungs %s, expected %s only"
                            % (rel(path), wrong, rung))
        rows = [r for r in rows if r.get("rung") == rung]
        keep = keep_analysable(rows, USABLE)
        mods = sorted({r["model"] for r in keep})
        inv.append([cond, rung, rel(path), len(rows), ";".join(mods),
                    "%d" % len(keep),
                    "ok" if keep else "empty"])
        if keep:
            AVAILABLE.append((cond, rung, mods))

show(["condition", "rung", "file", "rows", "models", "analysable", "state"],
     inv)
if problems:
    raise AssertionError("RUNG LABEL MISMATCH:\n  " + "\n  ".join(problems))
print()
print("PASS  every file on disk carries only the rung its name claims.")
print()
print("BASELINES, at N0 only: %s. No rung is planned in them; they are"
      % ", ".join(BASELINES))
print("here because a rung effect is only readable against the ceiling for")
print("its own condition. congruent_face is the ceiling for conflict_face,")
print("and both are the ceiling for anything the ladder does to dims.")
print()
print("The N0 rows come from Q1 and Q2. Every file here, baseline and rung")
print("alike, holds 612 distinct trials at %d repeats, so the second-order"
      % REPEATS)
print("contrasts below are read at the same resolution as Q1's and Q2's.")
print()
print("The X- rows are the off-ladder precedence directives, not rungs. They")
print("are read in cell 12b, against N0 and against N-A.")

condition       rung     file                                      rows  models                   analysable  state
--------------  -------  ----------------------------------------  ----  -----------------------  ----------  -----
dims            N0       runs/ex2_q1_dims_N0.jsonl                 612   claude_md;gemini;gpt_hi  576         ok   
dims            N-A      runs/ex2_q3_dims_N-A.jsonl                612   claude_md;gemini;gpt_hi  576         ok   
dims            N-C      runs/ex2_q3_dims_N-C.jsonl                612   claude_md;gemini;gpt_hi  576         ok   
dims            N-order  runs/ex2_q3_dims_N-order.jsonl            612   claude_md;gemini;gpt_hi  576         ok   
dims            N-D      runs/ex2_q3_dims_N-D.jsonl                612   claude_md;gemini;gpt_hi  576         ok   
dims            N-CD     runs/ex2_q3_dims_N-CD.jsonl               612   claude_md;gemini;gpt_hi  576         ok   
conflict_face   N0       runs/ex2_q2_conflict_face_N0.jsonl        612  

## Cell 12. Each rung against N0

No model calls. The contrast at each rung, and the change from `N0`, computed
**within condition** and never pooled across conditions: `C` supplies a missing
fact in `dims` and has to override a supplied one in `conflict`, so the two are
different manipulations wearing the same name.

In [22]:
# --- Cell 12. Rung contrasts against N0. No model calls. --------------------
contrast_rows = []
for cond, rung, mods in AVAILABLE:
    for model in mods:
        a = contrast_pairs(cond, rung, model)
        m_r, rlo, rhi, n_r = paired_mean_ci([d for _, d in a])
        if rung == "N0":
            contrast_rows.append([cond, model, rung, n_r, fmt(m_r), fmt(rlo),
                                  fmt(rhi), "-", "-", "-", "-"])
            continue
        b = contrast_pairs(cond, "N0", model)
        delta = paired_delta(a, b)
        mean, lo, hi, npos = paired_mean_ci([d for _, d in delta])
        contrast_rows.append([cond, model, rung, n_r, fmt(m_r), fmt(rlo),
                              fmt(rhi), npos, fmt(mean), fmt(lo), fmt(hi)])

show(["condition", "model", "rung", "npos", "contrast", "c_lo", "c_hi",
      "n_delta", "delta_vs_N0", "d_lo", "d_hi"], contrast_rows)
write_csv("tab_ex2_q3_contrasts.csv",
          ["condition", "model", "rung", "n_positions", "contrast_pts",
           "contrast_lo", "contrast_hi", "n_positions_delta",
           "delta_vs_N0_pts", "delta_lo", "delta_hi"], contrast_rows)
print()
print("contrast is the same small_minus_large Q1 and Q2 report, at that rung.")
print("delta_vs_N0 is how far the rung moved it, paired within position at")
print("both levels. A saturated rung has no useful interval on the contrast")
print("itself; read the delta.")

condition       model      rung     npos  contrast  c_lo    c_hi    n_delta  delta_vs_N0  d_lo   d_hi 
--------------  ---------  -------  ----  --------  ------  ------  -------  -----------  -----  -----
dims            claude_md  N0       32    -10.4     -24.3   3.5     -        -            -      -    
dims            gemini     N0       32    46.9      34.5    59.3    -        -            -      -    
dims            gpt_hi     N0       32    2.1       -6.2    10.3    -        -            -      -    
dims            claude_md  N-A      32    -14.6     -27.3   -1.8    32       -4.2         -23.3  15.0 
dims            gemini     N-A      32    96.9      92.4    101.4   32       50.0         37.6   62.4 
dims            gpt_hi     N-A      32    2.1       -3.7    7.9     32       0.0          -9.3   9.3  
dims            claude_md  N-C      32    0.0       0.0     0.0     32       10.4         -3.5   24.3 
dims            gemini     N-C      32    99.0      96.9    101.0   32   

## Cell 12b. The precedence directives, read

No model calls. The headline and its two controls in one table, with the
`congruent_face` `N0` ceiling beside them so the movement is read against how
far there was to move.

Each row is a paired within-position difference of the small-face against
large-face contrast, then differenced again against the comparison level.

In [23]:
# --- Cell 12b. The directive read-out. No model calls. ----------------------
DIRECTIVE_READS = (
    ("headline", "conflict_face", "X-image", "N0",
     "does the instruction move the model off the false stated face"),
    ("sentence only", "conflict_face", "X-image", "N-A",
     "subtracts being told to look; leaves the precedence sentence alone"),
    ("control: wording", "congruent_face", "X-image", "N0",
     "same prompt, antecedent never satisfied; movement here is not obedience"),
    ("control: symmetry", "conflict_face", "X-state", "N0",
     "one word different, still names the image; movement toward the picture "
     "here means the model is not reading which source was named"),
)

dir_rows = []
for label, cond, rung, against, why in DIRECTIVE_READS:
    for model in MODELS:
        a = contrast_pairs(cond, rung, model)
        b = contrast_pairs(cond, against, model)
        if not a or not b:
            dir_rows.append([label, cond, rung, against, model, 0, "-", "-",
                             "-", "not run"])
            continue
        mean, lo, hi, npos = paired_mean_ci([d for _, d in paired_delta(a, b)])
        # Keyed on npos first. An empty cell yields nan bounds, and
        # spans_zero counts nan as spanning, so a cell that was never run
        # would otherwise be reported as an interval consistent with no
        # effect -- a null read out of a file that does not exist.
        if not npos:
            interval = "not run"
        else:
            interval = "spans zero" if spans_zero(lo, hi) else "excludes zero"
        dir_rows.append([label, cond, rung, against, model, npos, fmt(mean),
                         fmt(lo), fmt(hi), interval])

show(["read", "condition", "rung", "vs", "model", "npos", "delta", "lo", "hi",
      "interval"], dir_rows)
write_csv("tab_ex2_q3_directive.csv",
          ["read", "condition", "rung", "against", "model", "npos", "delta",
           "lo", "hi", "interval"], dir_rows)

print()
print("THE CEILING, for scale. congruent_face at N0 is how far there was to")
print("move: the same prompt with a TRUE stated face, where no arbitration is")
print("required of the model at all.")
for model in MODELS:
    _c = contrast_pairs("congruent_face", "N0", model)
    m, lo, hi, n = paired_mean_ci([d for _, d in _c])
    print("  %-9s %s  [%s, %s]  over %d positions"
          % (model, fmt(m), fmt(lo), fmt(hi), n))
print()
for label, cond, rung, against, why in DIRECTIVE_READS:
    print("%-18s %s" % (label + ":", why))
print()
print("HOW TO READ THIS. The headline is only obedience if BOTH controls are")
print("flat. A wording control that moves says the model responded to having")
print("an extra sentence. A symmetry control that moves the same way says it")
print("responded to the word \"image\" rather than to which source was named.")
print()
print("LIMITATION, stated wherever this table is quoted. X uses the base")
print("schema, so the arm is committed before the opening is written. A model")
print("that did not move cannot be told apart from one that obeyed too late")
print("in the generation. Giving the directive the face-first schema would")
print("fix that and confound precedence with factor D, so it is not done.")

read               condition       rung     vs   model      npos  delta  lo     hi     interval     
-----------------  --------------  -------  ---  ---------  ----  -----  -----  -----  -------------
headline           conflict_face   X-image  N0   gpt_hi     32    88.0   72.3   103.8  excludes zero
headline           conflict_face   X-image  N0   gemini     32    200.0  200.0  200.0  excludes zero
headline           conflict_face   X-image  N0   claude_md  32    -22.9  -46.5  0.7    spans zero   
sentence only      conflict_face   X-image  N-A  gpt_hi     32    78.6   60.0   97.3   excludes zero
sentence only      conflict_face   X-image  N-A  gemini     32    196.9  193.5  200.3  excludes zero
sentence only      conflict_face   X-image  N-A  claude_md  32    -11.5  -35.3  12.3   spans zero   
control: wording   congruent_face  X-image  N0   gpt_hi     32    -22.4  -37.1  -7.7   excludes zero
control: wording   congruent_face  X-image  N0   gemini     32    0.0    0.0    0.0    span

## Cell 13. Attribution

No model calls.

**State this before reading the table.** These are differences of paired
differences. At 32 positions and one repeat the interval on one of them is
roughly 30 points whatever the data, so **small ladder effects are not
resolvable by this design**. A wide interval here is the design's resolution
showing, not evidence of no effect, and it must not be reported as a null.

- `N-D` minus `N-C` — does elicitation supply a missing fact, or force the
  application of one the model already held?
- `N-D` minus `N-order` — is `N-D`'s effect the instruction, or the field order
  it also changes?

In [24]:
# --- Cell 13. Attribution. No model calls. ----------------------------------
print("=" * 70)
print("RESOLUTION, stated before the numbers")
print("=" * 70)
print("These are differences of PAIRED DIFFERENCES. At %d positions and %d"
      % (len(USABLE), REPEATS))
print("repeat the interval on one is roughly %.0f points whatever the data." % GATE_MIN)
print("An interval that spans zero here means THE DESIGN CANNOT RESOLVE IT.")
print("It is not evidence of no effect and must not be written as one.")
print()

ATTRIB = (("N-D_minus_N-C", "N-D", "N-C",
           "does elicitation supply a missing fact, or force the "
           "application of one already held"),
          ("N-D_minus_N-order", "N-D", "N-order",
           "is N-D's effect the instruction, or the field order"))

have = {(c, r) for c, r, _ in AVAILABLE}
attrib_rows = []
for name, ra, rb, _q in ATTRIB:
    for cond in CONDITIONS:
        if (cond, ra) not in have or (cond, rb) not in have:
            continue
        for model in MODELS:
            a = contrast_pairs(cond, ra, model)
            b = contrast_pairs(cond, rb, model)
            if not any(d is not None for _, d in a) or \
               not any(d is not None for _, d in b):
                continue
            delta = paired_delta(a, b)
            mean, lo, hi, npos = paired_mean_ci([d for _, d in delta])
            verdict = ("NOT RESOLVABLE" if npos and hi - lo > 2 * GATE_MIN
                       and spans_zero(lo, hi)
                       else "positive" if lo > 0
                       else "negative" if hi < 0
                       else "spans zero")
            attrib_rows.append([cond, model, name, npos, fmt(mean), fmt(lo),
                                fmt(hi), verdict])

if not attrib_rows:
    print("Neither rung pair is on disk yet, so there is nothing to")
    print("attribute. Cell 9 buys N-C and N-order.")
else:
    show(["condition", "model", "contrast", "npos", "mean", "lo", "hi",
          "reading"], attrib_rows)
    write_csv("tab_ex2_q3_attribution.csv",
              ["condition", "model", "contrast", "n_positions", "mean_pts",
               "paired_lo", "paired_hi", "reading"], attrib_rows)
    print()
    for name, ra, rb, q in ATTRIB:
        print("  %-18s %s" % (name, q))

RESOLUTION, stated before the numbers
These are differences of PAIRED DIFFERENCES. At 32 positions and 3
repeat the interval on one is roughly 30 points whatever the data.
An interval that spans zero here means THE DESIGN CANNOT RESOLVE IT.
It is not evidence of no effect and must not be written as one.



condition      model      contrast           npos  mean   lo     hi    reading   
-------------  ---------  -----------------  ----  -----  -----  ----  ----------
dims           gpt_hi     N-D_minus_N-C      32    3.1    -9.8   16.0  spans zero
dims           gemini     N-D_minus_N-C      32    1.0    -1.0   3.1   spans zero
dims           claude_md  N-D_minus_N-C      32    0.0    -13.1  13.1  spans zero
conflict_face  gpt_hi     N-D_minus_N-C      32    22.9   12.2   33.7  positive  
conflict_face  gemini     N-D_minus_N-C      32    0.0    0.0    0.0   spans zero
conflict_face  claude_md  N-D_minus_N-C      32    50.0   35.6   64.4  positive  
dims           gpt_hi     N-D_minus_N-order  32    2.1    -11.0  15.2  spans zero
dims           gemini     N-D_minus_N-order  32    54.2   42.2   66.2  positive  
dims           claude_md  N-D_minus_N-order  32    -6.2   -27.5  15.0  spans zero
conflict_face  gpt_hi     N-D_minus_N-order  32    20.8   12.2   29.5  positive  
conflict_face  g

## Cell 14. Does the reported opening improve with rung

No model calls. `opening_needed_m` is a self-report and never a scored endpoint
on its own, but it localises the failure: a rung that fixes the arm without
fixing the reported opening is doing something other than what it claims.

In [25]:
# --- Cell 14. Reported opening by rung. No model calls. ---------------------
TOL = 0.006          # grade.classify_width's tolerance, not a new one
rep_rows = []
for cond, rung, mods in AVAILABLE:
    for model in mods:
        for face in FACES:
            sub = [r for r in rung_rows_for(cond, rung)
                   if r["model"] == model and r["face"] == face]
            stated = [r for r in sub if r.get("opening_needed_m") is not None]
            true_open = FACTS[face]["grasp_m"]
            right = sum(1 for r in stated
                        if abs(r["opening_needed_m"] - true_open) <= TOL)
            lo, hi = wilson(right, len(stated))
            rep_rows.append([cond, model, rung, face, len(sub), len(stated),
                             right, fmt(pct(right, len(stated))), fmt(lo),
                             fmt(hi)])

show(["condition", "model", "rung", "face", "trials", "stated", "correct",
      "pct", "lo", "hi"], rep_rows)
write_csv("tab_ex2_q3_reported.csv",
          ["condition", "model", "rung", "resting_face", "n_trials",
           "n_stated", "n_correct", "correct_pct", "wilson_lo", "wilson_hi"],
          rep_rows)
print()
print("In CONFLICT the true opening is not the declared one, so a low score")
print("here is a model believing the text, not a model failing to derive.")
print("Cross-read it with Q2's tab_ex2_q2_source.csv before calling it an")
print("error.")

condition       model      rung     face        trials  stated  correct  pct    lo    hi   
--------------  ---------  -------  ----------  ------  ------  -------  -----  ----  -----
dims            claude_md  N0       small_face  96      96      54       56.2   46.3  65.7 
dims            claude_md  N0       large_face  96      96      35       36.5   27.5  46.4 
dims            gemini     N0       small_face  96      96      94       97.9   92.7  99.4 
dims            gemini     N0       large_face  96      96      47       49.0   39.2  58.8 
dims            gpt_hi     N0       small_face  96      96      91       94.8   88.4  97.8 
dims            gpt_hi     N0       large_face  96      96      7        7.3    3.6   14.3 
dims            claude_md  N-A      small_face  96      96      34       35.4   26.6  45.4 
dims            claude_md  N-A      large_face  96      96      47       49.0   39.2  58.8 
dims            gemini     N-A      small_face  96      96      96       100.0  

## Cell 15. The face the model names

No model calls, and **only `N-D` and `N-CD` ask for it**, because those are the
rungs whose schema is `face_first`.

Reported separately from the opening, because a model that names the face
correctly and still gives the wrong opening has failed at the derivation, while
one that names it wrongly has failed at the perception. Q1 could not see this
distinction: no rung it ran required the face.

In [26]:
# --- Cell 15. Face-report accuracy. No model calls. -------------------------
FACE_RUNGS = tuple(r for r in LADDER
                   if P.RUNGS[r]["schema"] == "face_first")
print("rungs whose schema asks for the face: %s" % ", ".join(FACE_RUNGS))
print()
face_rows = []
for cond, rung, mods in AVAILABLE:
    if rung not in FACE_RUNGS:
        continue
    for model in mods:
        sub = [r for r in rung_rows_for(cond, rung) if r["model"] == model]
        named = [r for r in sub if r.get("resting_face")]
        right = sum(1 for r in named if r["resting_face"] == r["true_pose"])
        lo, hi = wilson(right, len(named))
        # The cell that matters: face right, opening wrong.
        split = sum(1 for r in named
                    if r["resting_face"] == r["true_pose"]
                    and r.get("opening_needed_m") is not None
                    and abs(r["opening_needed_m"]
                            - FACTS[r["true_pose"]]["grasp_m"]) > 0.006)
        face_rows.append([cond, model, rung, len(sub), len(named), right,
                          fmt(pct(right, len(named))), fmt(lo), fmt(hi),
                          split])

if not face_rows:
    print("No face_first rung is on disk yet. Cell 6 buys %s." % GATE_RUNG)
else:
    show(["condition", "model", "rung", "trials", "named", "correct", "pct",
          "lo", "hi", "face_right_open_wrong"], face_rows)
    write_csv("tab_ex2_q3_face.csv",
              ["condition", "model", "rung", "n_trials", "n_named",
               "n_correct", "correct_pct", "wilson_lo", "wilson_hi",
               "n_face_right_opening_wrong"], face_rows)
    print()
    print("face_right_open_wrong is the diagnostic cell. A model there SAW")
    print("the orientation and still could not turn it into an opening,")
    print("which is a derivation failure. A low correct_pct instead is a")
    print("perception failure, and the two want different remedies.")

rungs whose schema asks for the face: N-D, N-CD

condition      model      rung  trials  named  correct  pct    lo    hi     face_right_open_wrong
-------------  ---------  ----  ------  -----  -------  -----  ----  -----  ---------------------
dims           claude_md  N-D   192     192    97       50.5   43.5  57.5   31                   
dims           gemini     N-D   192     192    192      100.0  98.0  100.0  0                    
dims           gpt_hi     N-D   192     192    109      56.8   49.7  63.6   38                   
dims           claude_md  N-CD  192     192    97       50.5   43.5  57.5   5                    
dims           gemini     N-CD  192     192    192      100.0  98.0  100.0  0                    
dims           gpt_hi     N-CD  192     192    127      66.1   59.2  72.5   0                    
conflict_face  claude_md  N-D   192     192    0        0.0    0.0   2.0    0                    
conflict_face  gemini     N-D   192     192    0        0.0    0.0   

## Cell 16. Provenance

No model calls. Every file this notebook read, with its row count and hash, the
prompt version, the models and the date.

In [27]:
# --- Cell 16. Provenance. No model calls. -----------------------------------
prov = []
today = datetime.date.today().isoformat()
prov.append(provenance_row("captures", CAPTURES / "consults.jsonl", OUT,
                           today=today,
                           default_version=P.EX2_PROMPT_VERSION))
for cond in CONDITIONS:
    for rung in LADDER:
        path = N0_FILE[cond] if rung == "N0" else rung_file(cond, rung)
        if pathlib.Path(path).exists():
            prov.append(provenance_row("%s_%s" % (cond, rung), path, OUT,
                                       today=today,
                                       default_version=P.EX2_PROMPT_VERSION))
for cond, rung in DIRECTIVE_CELLS:
    path = rung_file(cond, rung)
    if pathlib.Path(path).exists():
        prov.append(provenance_row("%s_%s" % (cond, rung), path, OUT,
                                   today=today,
                                   default_version=P.EX2_PROMPT_VERSION))
if pathlib.Path(CONTROL_OUT).exists():
    prov.append(provenance_row("dims_%s_noimage" % GATE_RUNG, CONTROL_OUT,
                               OUT, today=today,
                               default_version=P.EX2_PROMPT_VERSION))

show(["role", "rows", "sha256", "prompt_version", "models"],
     [[r[0], r[2], (r[3] or "")[:12], r[4], r[5]] for r in prov])
write_csv("tab_ex2_q3_provenance.csv",
          ["role", "path", "rows", "sha256", "prompt_version", "model_string",
           "run_date"], prov)

print()
print("DESIGN FACTS THAT MUST BE DISCLOSED IN THE CHAPTER")
print("-" * 70)
print("1. The gate rung is %s, not N-CD. The 2026-08-27 pilots already showed"
      % GATE_RUNG)
print("   gemini going from 46.9 at N0 to a complete flip under N-D alone,")
print("   with the face named correctly on every trial, and gpt not moving,")
print("   so elicitation is the rung the evidence implicates. N-CD is a")
print("   sufficiency cell for a model N-D does not move, never an")
print("   interaction test: this design is not powered for one.")
print("2. Rungs and baselines alike run at %d repeats over 612 trials, so an"
      % REPEATS)
print("   interval spanning zero is the design's resolution, not evidence")
print("   of no effect.")
print("3. X-image and X-state are NOT rungs. They break the boundary rule the")
print("   four factors are held to -- they name the stated resting face and")
print("   say the image can contradict it -- so they are off the ladder, they")
print("   are refused in dims, and they answer a different question: whether")
print("   a model has an arbitration step an instruction can reach at all.")
print("   Cell 4b prints their wording; cell 12b reads them.")
print("3. Contrasts are computed WITHIN condition and never pooled across")
print("   them: C supplies a missing fact in dims and overrides a supplied")
print("   one in conflict, so it is two manipulations under one name.")
print("4. N-order is the control for N-D. N-D changes the wording AND the")
print("   field order, and a model generates left to right, so without the")
print("   order control an N-D effect could not be attributed to either.")
print("5. resting_face is asked for only at %s. Asking at every rung would"
      % ", ".join(FACE_RUNGS))
print("   tell the model the face matters, which is what factor D")
print("   manipulates.")
print("6. The dims gate file is SEEDED from two 2026-08-27 pilot files,")
print("   runs/ex2_q1_dims_N-D_effort.jsonl and ..._N-D_order.jsonl, which")
print("   hold this exact cell for gpt_hi and gemini at the same prompt")
print("   version. Those rows were collected on 2026-08-27, not in this")
print("   sweep, and the seeding filter rebuilds the trial_id this cell")
print("   would ask for so that no other factor arm can enter. The row")
print("   count above therefore mixes two collection dates, which is what")
print("   resuming a run always does and is why the hash is recorded.")
print("7. Stage 3 is a PROBE plus a gated expansion, not the full ladder.")
print("   N-CD runs in conflict for all three models; the other four rungs")
print("   are bought only for models it moved. Buying all five for all")
print("   three would have been 1,020 calls, 65 percent of the notebook,")
print("   spent before knowing whether any instruction moves anything in a")
print("   condition Q2 measured at -100 points unanimously.")
print("8. Prompt version %s. Preference %s. %d usable positions."
      % (P.EX2_PROMPT_VERSION, PREFERENCE, len(USABLE)))

role                    rows  sha256        prompt_version  models                 
----------------------  ----  ------------  --------------  -----------------------
captures                102   b1579d5d8d1f  2026-08-27b                            
dims_N0                 615   01a58e0f7557  2026-08-27b     claude_md;gemini;gpt_hi
dims_N-A                612   35b4a968c8cc  2026-08-27b     claude_md;gemini;gpt_hi
dims_N-C                612   67c89efcedda  2026-08-27b     claude_md;gemini;gpt_hi
dims_N-order            613   2fb87a1e9970  2026-08-27b     claude_md;gemini;gpt_hi
dims_N-D                614   67d9955d92db  2026-08-27b     claude_md;gemini;gpt_hi
dims_N-CD               612   aaa13ffc0bb1  2026-08-27b     claude_md;gemini;gpt_hi
conflict_face_N0        612   bf1ca02d9b59  2026-08-27b     claude_md;gemini;gpt_hi
conflict_face_N-A       613   0f6f6bab61fe  2026-08-27b     claude_md;gemini;gpt_hi
conflict_face_N-C       612   e365876eca16  2026-08-27b     claude_md;gemini